# U04 資料庫設計：ER 模型與正規化

**資料庫管理**・10/01　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

從「情境訪談」推到「正確的 schema」——今天的工作坊就是**你自己專題的第一塊**

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 |
|---|---|---|
| 第 1 節 | 50 | 設計流程・ER 模型與基數・**案例：把「社團活動報名」從訪談推到 DDL**・更多題型的 ER 速寫・**七大建模模式（主線跑①②③⑤；④⑥⑦選讀）** |
| 第 2 節 | 50 | 爛表的三種異常現場・FD 與 closure・1NF→3NF・無損驗證・反正規化與星型；**BCNF／3NF 演算法與 MVD／4NF 為進階選讀** |
| 工作坊 | 35 | **為自己被指派的題目**做名詞動詞分析 → ER → DDL 草稿，互檢＋自檢器 |

> 為什麼要學設計？**schema 是應用的地基**：地基歪了，之後每一個功能都在跟它打架。
> 改一張表 = 改所有用到它的查詢與介面；第 4 個單元改很便宜，第 14 週改會想哭。
>
> **50 分鐘主線**停在「一表一主題、closure 找 key、逐條 FD 抽表、無損驗證」；§2.7–2.8 保留給想看演算法保證的同學選讀。

# 第 1 節：ER 模型——先想清楚，再開表

## 1.1 設計的標準流程

```
需求（訪談、情境） ──► 概念設計（ER 圖） ──► 邏輯設計（關聯綱要） ──► 物理實作（DDL＋索引）
      U04                   U04                    U04                工作坊／U07
```

直接跳到開表（很多人、很多 AI 的壞習慣）會發生什麼？
- 想到什麼欄就加什麼 → 一張 20 欄的巨表 → 更新異常天天發生（第 2 節示範）
- 沒想清楚「一個 X 能對幾個 Y」→ 名額、重複報名、時段衝突全靠 Python if 硬擋 → 擋不住（U06 證明給你看）

## 1.2 ER 三要素

| 要素 | 是什麼 | 圖上長相 | 例 |
|---|---|---|---|
| **實體 Entity** | 值得單獨記錄的「東西」 | 方框 | 會員、社團、活動 |
| **屬性 Attribute** | 實體的性質 | 橢圓／列在框內 | 姓名、名額、日期 |
| **關係 Relationship** | 實體之間的動詞 | 菱形／連線 | 會員「報名」活動、社團「舉辦」活動 |

畫圖工具自由：紙筆拍照、[dbdiagram.io](https://dbdiagram.io)（推薦，打字就出圖）、draw.io。**專題要附一張**（共同要求第 1 條）。

### 工具速報：dbdiagram.io 三分鐘上手（工作坊直接用）

左邊打字、右邊出圖，匯出 PNG 貼進報告。語法就三招：

```
Table member {
  member_id int [pk]
  name      varchar [not null]
  phone     varchar [unique]
}
Table registration {
  reg_id    int [pk]
  member_id int [ref: > member.member_id]   // N:1 —— > 指向「一」的那邊
  event_id  int [ref: > event.event_id]
  status    varchar
}
```

`>` 多對一、`<` 一對多、`-` 一對一。畫完按 Export → PNG。（紙筆手繪拍照也完全可以——重點是圖上有基數。）

## 1.3 基數（cardinality）：設計的靈魂拷問

判斷心法——**兩個方向各問一次**：「一個 X 最多對幾個 Y？一個 Y 最多對幾個 X？」

| 答案組合 | 關係 | 例 | 實作方式 |
|---|---|---|---|
| 1 對 1 | 1:1 | 學生↔學號卡 | 合併成一表，或 FK＋UNIQUE |
| 1 對 多 | 1:N | 社團→活動（一社多活動；一活動屬一社） | **N 方放 FK** |
| 多 對 多 | M:N | 會員↔活動（互相都可多個） | **關聯表**（junction table） |

再多問一句「**可以是 0 嗎？**」（participation）：活動可以沒人報名嗎？會員可以不屬於任何社團嗎？——答案決定 `NOT NULL` 與 LEFT JOIN 的寫法。

### 隨堂練習 A（3 分鐘口頭）
1. 民宿：房間 vs 訂單？　2. 電影院：場次 vs 座位票？　3. 問卷：問題 vs 選項？
4. 獸醫診所：寵物 vs 飼主？　5. 進銷存：進貨單 vs 商品？

<details><summary>答案</summary>
1. 1:N（一房多訂單；訂單日期不重疊是額外的「約束」，不是基數）　2. 1:N（一場次多張票；一張票一場次一座位——(場次,座位) 要 UNIQUE）　3. 1:N　4. N:1（多寵一主；若允許共同飼養就是 M:N——**跟業主確認，這就是需求訪談**）　5. M:N（透過進貨明細表）
</details>

In [ ]:
# 基數的三種長相，各寫一次最小 DDL——1:1 用「FK＋UNIQUE」鎖死
import sqlite3
one = sqlite3.connect(":memory:")
one.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE student(sid TEXT PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE locker(                                   -- 置物櫃：一人最多一櫃、一櫃最多一人
  locker_id INTEGER PRIMARY KEY,
  sid TEXT UNIQUE REFERENCES student(sid));            -- ← FK＋UNIQUE ＝ 1:1
INSERT INTO student VALUES ('S001','林佳蓉'),('S002','陳威廷');
INSERT INTO locker(sid) VALUES ('S001');
""")
try:
    one.execute("INSERT INTO locker(sid) VALUES ('S001')")     # S001 想佔第二櫃
except sqlite3.IntegrityError as e:
    print("✅ 1:1 被 UNIQUE 守住 →", e)
print("→ 1:N 就是把 UNIQUE 拿掉；M:N 另開關聯表。基數不是畫圖的裝飾，是「約束的選擇」。")

### 代理鍵 vs 自然鍵：什麼時候敢用「業務本來的編號」？

| | 自然鍵（學號、ISBN） | 代理鍵（自動編號） |
|---|---|---|
| 前提 | **真的永遠唯一、永遠不變** | 無 |
| 好處 | 可讀、防重複天生 | 穩定、join 快、改業務規則不痛 |
| 風險 | 「不變」常是幻覺（換學號、ISBN 再版） | 要另設 UNIQUE 擋業務重複 |

實務預設：**代理鍵當 PK＋業務編號設 UNIQUE**——兩個好處都拿。你的專題照這個寫最穩。

### 補充概念：弱實體（weak entity）——「活在別人身上」的資料

**訂單明細**沒有自己的獨立身分：離開訂單，「第 2 列」毫無意義。這種實體叫**弱實體**：

- 識別靠「爸爸的主鍵＋自己的序號」：`order_item(order_id, line_no)` 複合主鍵；
- 生死與共：實作上就是 `FK + ON DELETE CASCADE`（上個單元的連鎖刪除，正當用途在這）；
- 你的題目裡到處都是：訂單明細、病歷、比賽數據紀錄、問卷填答。

（用代理鍵 `item_id INTEGER PRIMARY KEY` ＋ `UNIQUE(order_id, line_no)` 也行——兩派都對，講得出理由就好。）

In [ ]:
# 弱實體的實作長相：複合主鍵＋CASCADE——爸爸走了，明細跟著走
we = sqlite3.connect(":memory:")
we.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE orders(order_id INTEGER PRIMARY KEY, odate TEXT NOT NULL);
CREATE TABLE order_item(
  order_id INTEGER NOT NULL REFERENCES orders(order_id) ON DELETE CASCADE,
  line_no  INTEGER NOT NULL,
  pname    TEXT NOT NULL,
  qty      INTEGER NOT NULL CHECK (qty > 0),
  PRIMARY KEY (order_id, line_no));                    -- 識別＝爸爸的 PK＋自己的序號
INSERT INTO orders(order_id, odate) VALUES (1, '2026-10-01');
INSERT INTO order_item VALUES (1, 1, '珍珠奶茶', 2), (1, 2, '蛋餅', 1);
""")
we.execute("DELETE FROM orders WHERE order_id = 1")
print("訂單刪除後，明細剩", we.execute("SELECT COUNT(*) FROM order_item").fetchone()[0],
      "列——弱實體與爸爸生死與共（上個單元 CASCADE 的正當用途）")

## 1.4 案例研究：社團活動報名系統（從訪談逐字稿開始）

> 「我們**社團**很多，每個社團整學期會辦好幾場**活動**。**會員**看到活動就能**報名**，
> 但活動有**名額**上限，滿了要排**候補**；報名有**截止時間**。一個人同一場活動只能報一次，
> 但可以報很多不同活動。喔對，活動要記在哪個教室、什麼時候辦。」

**名詞→實體候選，動詞→關係候選**（設計的第一招）：

- 名詞：社團、活動、會員、名額、候補、截止時間、教室 → 實體：**member、club、event**；「名額／截止／教室」是 event 的屬性；「候補」是報名的**狀態**不是實體
- 動詞：舉辦（club—event，1:N）、報名（member—event，**M:N＋自帶屬性**：狀態、時間）

**M:N 關係自帶屬性 → 它自己就該是一張表**：`registration`。這是所有應用系統最常見的 pattern（訂單明細、選課、借閱⋯⋯你的題目裡一定有）。

## 1.5 報名系統的 ER 圖與轉表三規則

```
 club ──1───舉辦───N── event ──N───報名───M── member
 (社團)               (活動)      │           (會員)
                                registration（關係實體：狀態、報名時間）
```

**ER → 關聯綱要三規則**：
1. 每個實體 → 一張表，主鍵伺候（代理鍵 `INTEGER PRIMARY KEY` 是安全預設）。
2. 1:N 關係 → **N 方**加 FK（event 加 `club_id`）。
3. M:N 關係 → 獨立關聯表，**兩支 FK**＋自身屬性；用 `UNIQUE(兩支FK)` 或複合主鍵表達「一人一活動一筆」。

下一格是完整 DDL——注意每一條約束都對應訪談裡的一句話：

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("club.db")
con.executescript("""
PRAGMA foreign_keys = ON;
DROP TABLE IF EXISTS registration; DROP TABLE IF EXISTS event;
DROP TABLE IF EXISTS member;       DROP TABLE IF EXISTS club;

CREATE TABLE club(
  club_id INTEGER PRIMARY KEY,
  cname   TEXT NOT NULL UNIQUE,
  office  TEXT);

CREATE TABLE member(
  member_id INTEGER PRIMARY KEY,
  name  TEXT NOT NULL,
  phone TEXT UNIQUE);                          -- 「電話不能重複註冊」

CREATE TABLE event(
  event_id INTEGER PRIMARY KEY,
  club_id  INTEGER NOT NULL REFERENCES club(club_id),   -- 規則 2：1:N 的 N 方
  title    TEXT NOT NULL,
  room     TEXT,
  held_at  TEXT NOT NULL,                      -- 「什麼時候辦」
  quota    INTEGER NOT NULL CHECK (quota > 0), -- 「名額上限」
  deadline TEXT NOT NULL);                     -- 「報名截止」

CREATE TABLE registration(                     -- 規則 3：M:N ＋ 自帶屬性
  reg_id    INTEGER PRIMARY KEY,
  member_id INTEGER NOT NULL REFERENCES member(member_id),
  event_id  INTEGER NOT NULL REFERENCES event(event_id),
  status    TEXT NOT NULL DEFAULT '報名'
            CHECK (status IN ('報名','候補','取消')),   -- 「候補」是狀態
  reg_time  TEXT NOT NULL DEFAULT (datetime('now','+8 hours')),
  UNIQUE (member_id, event_id));               -- 「同一活動只能報一次」
""")
con.executemany("INSERT INTO club(cname, office) VALUES (?,?)",
                [("資料科學社","理學院 302"), ("登山社","體育館 B1")])
con.executemany("INSERT INTO member(name, phone) VALUES (?,?)",
                [("林佳蓉","0912-111-111"), ("陳威廷","0912-222-222"), ("張雅筑","0912-333-333")])
con.executemany("INSERT INTO event(club_id,title,room,held_at,quota,deadline) VALUES (?,?,?,?,?,?)", [
    (1,"Python 資料分析工作坊","電腦教室 1","2026-10-20 18:30",2,"2026-10-18"),
    (1,"SQL 快打旋風","電腦教室 2","2026-11-03 18:30",30,"2026-11-01"),
    (2,"合歡山北峰行前會","社辦","2026-10-25 19:00",15,"2026-10-23")])
con.executemany("INSERT INTO registration(member_id, event_id) VALUES (?,?)",
                [(1,1),(2,1),(3,2)])
con.commit()
print("報名系統 schema 就緒 ✅（4 表、3 FK、6 種約束都用上了）")

In [ ]:
# schema 好不好，看它「答不答得出核心問題」——剩餘名額（之後 App 的心臟查詢）
q = lambda sql, p=(): pd.read_sql_query(sql, con, params=p)
q("""SELECT e.event_id, c.cname AS 社團, e.title AS 活動, e.quota AS 名額,
        COUNT(r.reg_id) AS 已報,
        e.quota - COUNT(r.reg_id) AS 剩餘
     FROM event e
     JOIN club c ON e.club_id = c.club_id
     LEFT JOIN registration r
            ON e.event_id = r.event_id AND r.status = '報名'   -- 右表條件放 ON（上個單元的陷阱！）
     GROUP BY e.event_id""")

In [ ]:
# 約束替你守住訪談裡的每一句話
tests = [
    ("同一人重複報名",  "INSERT INTO registration(member_id, event_id) VALUES (1, 1)"),
    ("幽靈活動",        "INSERT INTO registration(member_id, event_id) VALUES (1, 999)"),
    ("名額開 0",        "INSERT INTO event(club_id,title,held_at,quota,deadline) VALUES (1,'x','2026-11-11',0,'2026-11-09')"),
    ("亂寫狀態",        "UPDATE registration SET status = '已讀' WHERE reg_id = 1"),
]
for label, sql in tests:
    try:
        con.execute(sql); print(f"⚠️ {label}：過了？！")
    except sqlite3.IntegrityError as e:
        print(f"✅ {label:10s} 擋下 → {e}")
con.commit()

In [ ]:
# 「可以是 0 嗎？」（participation）——吉他社還沒辦活動，報表也不能漏掉它
con.execute("INSERT OR IGNORE INTO club(cname, office) VALUES ('吉他社', '藝文中心 2F')")
con.commit()
q("""SELECT c.cname AS 社團, COUNT(e.event_id) AS 活動數
     FROM club c LEFT JOIN event e ON c.club_id = e.club_id
     GROUP BY c.club_id ORDER BY 活動數 DESC""")
# participation「可為 0」的那一邊，報表要用 LEFT JOIN 才不會人間蒸發（上個單元的老朋友）

### 隨堂練習 B2（2 分鐘，動手）：反方向的 participation

反過來問：「**還沒報名任何活動**的會員」——上個單元的 anti-join 直接派上用場。先想：左表是誰？

<details><summary>參考解</summary>

```sql
SELECT m.member_id, m.name
FROM member m LEFT JOIN registration r ON m.member_id = r.member_id
WHERE r.reg_id IS NULL;
```
（或 `NOT EXISTS`。）「找沒有的」永遠是 LEFT JOIN + IS NULL／NOT EXISTS——催報名名單、未繳費名單都是它。
</details>

In [ ]:
# 練習 B2 工作區（club.db 還開著，直接查）
# TODO




## 1.6 再練三題：不同題型的 ER 速寫

### 民宿訂房（「區間」型）
> 「客人訂**某房型**的房，**幾月幾號住到幾月幾號**；同一間房同期間不能給兩組人；房價隨季節變。」

- 實體：room_type、room（1 房型 N 房）、guest、booking；**rate**（房型×起始日→價格，見 1.7 模式⑥）
- 「同房不重疊」不是基數，是**約束＋查詢**（區間重疊判斷，1.7 模式②）
- booking 的 FK 指 room 還是 room_type？——訪談問清楚：「客人訂的是『某一間』還是『某一種』？」兩種都是合法設計，系統行為不同。

### 電影院訂票（「座位唯一」型）
> 「一場**放映**在某**影廳**，廳裡有固定**座位**；一張**票**＝某場次的某座位，**不能賣兩次**。」

- 實體：movie、hall、seat（屬於 hall）、showtime（movie×hall×時間）、ticket
- 靈魂約束：`UNIQUE(showtime_id, seat_id)`——下一格直接寫出來踩踩看。

### 問卷平台（「一題一答一列」型）
> 「一份**問卷**很多**題**，單選題有**選項**；**填答者**每題答一次，同一份問卷不能重複填。」

- 實體：survey、question（1:N）、choice（1:N）、respondent、**response（一題一答一列！）**
- 常見錯誤：把 20 題做成 response 表的 20 個欄位——題目一改 schema 就重蓋；「一題一答一列」才進得了統計分析（long format，統計系你懂的）。

In [ ]:
# 訂房系統的核心查詢先寫給你：「跨年夜還有哪些雙人房？」——區間重疊 × NOT EXISTS
t14 = sqlite3.connect(":memory:")
t14.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE room(room_id INTEGER PRIMARY KEY, rname TEXT NOT NULL, room_type TEXT NOT NULL);
CREATE TABLE booking(
  booking_id INTEGER PRIMARY KEY,
  room_id INTEGER NOT NULL REFERENCES room(room_id),
  check_in TEXT NOT NULL, check_out TEXT NOT NULL,     -- 住 [check_in, check_out)
  CHECK (check_in < check_out));
INSERT INTO room(rname, room_type) VALUES ('201','雙人房'),('202','雙人房'),('301','四人房');
INSERT INTO booking(room_id, check_in, check_out) VALUES
  (1, '2026-12-30', '2027-01-02');                     -- 201 被跨年訂單卡住
""")
arrive, depart = '2026-12-31', '2027-01-01'
df14 = pd.read_sql_query("""
    SELECT r.rname AS 房號, r.room_type AS 房型 FROM room r
    WHERE r.room_type = '雙人房'
      AND NOT EXISTS (SELECT 1 FROM booking b
                      WHERE b.room_id = r.room_id
                        AND b.check_in < ? AND ? < b.check_out)   -- 重疊判斷（模式②）
    ORDER BY r.rname""", t14, params=(depart, arrive))
print(f"{arrive} 入住一晚，可訂的雙人房：\n" + df14.to_string(index=False))
print("→ 201 被卡、202 可訂。預約、掛號、訂房類題目的「查空檔」全是這句的變形。")

In [ ]:
# 訂票系統的靈魂約束現場：同一場次同一座位，第二張票進不來
con.executescript("""
DROP TABLE IF EXISTS ticket; DROP TABLE IF EXISTS showtime;
CREATE TABLE showtime(showtime_id INTEGER PRIMARY KEY, movie TEXT NOT NULL, starts TEXT NOT NULL);
CREATE TABLE ticket(
  ticket_id   INTEGER PRIMARY KEY,
  showtime_id INTEGER NOT NULL REFERENCES showtime(showtime_id),
  seat_no     TEXT NOT NULL,
  UNIQUE (showtime_id, seat_no));              -- ← 一位一票，資料庫層保證
INSERT INTO showtime(movie, starts) VALUES ('沙丘三', '2026-10-01 19:30');
INSERT INTO ticket(showtime_id, seat_no) VALUES (1, 'A1'), (1, 'A2');
""")
try:
    con.execute("INSERT INTO ticket(showtime_id, seat_no) VALUES (1, 'A1')")   # A1 再賣一次！
except sqlite3.IntegrityError as e:
    print("✅ 重複售位被擋 →", e)
con.commit()
print(q("SELECT * FROM ticket").to_string(index=False))
print("→ U06 會看到：兩個人「同一瞬間」搶 A1，Python 的 if 擋不住、這條 UNIQUE 擋得住。")

In [ ]:
# 問卷平台的靈魂：「一題一答一列」（long format）——存 long、看 wide，統計系的主場
con.executescript("""
DROP TABLE IF EXISTS response;
CREATE TABLE response(
  respondent_id INTEGER NOT NULL,
  question_id   INTEGER NOT NULL,
  answer        TEXT NOT NULL,
  PRIMARY KEY (respondent_id, question_id));     -- 每人每題一列（順便防同題重複作答）
INSERT INTO response VALUES
  (1, 1, '非常同意'), (1, 2, '每天'), (1, 3, '5'),
  (2, 1, '同意'),     (2, 2, '偶爾'), (2, 3, '4'),
  (3, 1, '非常同意'), (3, 2, '每天'), (3, 3, '5');
""")
con.commit()
print("存的時候是 long：")
print(q("SELECT * FROM response LIMIT 4").to_string(index=False))
print("\n看的時候用條件式聚合攤成 wide（上個單元的樞紐技）：")
print(q("""SELECT respondent_id AS 填答者,
             MAX(CASE WHEN question_id = 1 THEN answer END) AS Q1,
             MAX(CASE WHEN question_id = 2 THEN answer END) AS Q2,
             MAX(CASE WHEN question_id = 3 THEN answer END) AS Q3
          FROM response GROUP BY respondent_id""").to_string(index=False))
print("\n→ 問卷加第 4 題：long 版 schema 一個字不用改；「20 題 20 欄」版要 ALTER 重蓋還丟歷史。")

### 隨堂練習 B：基數判斷 8 連發（同桌互考）

各是 1:1／1:N／M:N？實作時 FK 放哪（或要不要關聯表）？

1. 場地預約：場地 vs 預約　2. 家教媒合：老師 vs 可授課時段　3. 售票：訂單 vs 票種（透過什麼？）
4. 進銷存：商品 vs 供應商　5. 健身房：會員 vs 會籍（membership）　6. 診所：獸醫 vs 掛號
7. 志工系統：志工 vs 服務活動　8. 選課系統：學生 vs 志願單上的課

<details><summary>答案</summary>

1. 1:N（booking 放 room_id）　2. 1:N（slot 放 tutor_id）　3. M:N——經 order_item（訂單明細，弱實體）
4. 看業務！單一供應商＝1:N（product 放 supplier_id）；可多供應商＝M:N（product_supplier 表）——**去問業主**
5. 1:N（一人可續約多期，membership 放 member_id；「同一時間只一張有效」是額外約束）
6. 1:N（appointment 放 vet_id；「同醫同時段唯一」用 UNIQUE(vet_id, at)）
7. M:N（signup／hour_log 關聯表）　8. M:N（wishlist 關聯表＋志願序屬性）
</details>

### 隨堂練習 C（3 分鐘，動手）：借書處的兩句訪談

> 「**讀者**辦證後可以**借書**；每本**書**有好幾**冊**；一冊一次只能被一個人借走，還了才能再借。」

名詞動詞分析 → 實體、關係、基數？「一冊一次只能被一人借走」是基數還是約束？

<details><summary>參考</summary>

實體：reader、book（書目）、copy（冊，1 書 N 冊）、loan（reader×copy 的 M:N＋時間屬性）。
「一冊同時只能在一人手上」＝**約束**（同 copy 的未歸還 loan 至多一筆——部分唯一索引或查詢檢查），不是基數（歷史上一冊可被很多人借過）。
「當下」與「歷史」的區別，是設計訪談最常見的澄清點。
</details>

In [ ]:
# 練習 C 延伸工作區：把借書處的 schema 骨架寫出來（4 張表，先只求 PK/FK 正確）
lib = sqlite3.connect(":memory:")
lib.executescript("""
PRAGMA foreign_keys = ON;
-- TODO: reader / book / copy / loan
-- CREATE TABLE ...;

""")
print("你的表：", [r[0] for r in lib.execute("SELECT name FROM sqlite_master WHERE type='table'")])
# 寫完自問：loan 的 FK 指 book 還是 copy？（借的是「那一冊」！）——U05 示範專題就是這個系統的完全體

### ER 圖自我審查 5 問（畫完先自問，再給別人看）

1. 每個實體都講得出「一列代表一個＿＿」嗎？
2. 每條關係的**兩個方向**都問過「最多幾個？可為 0 嗎？」了嗎？
3. M:N 的關係屬性（狀態、時間、數量）都掛在**關聯表**上了嗎（不是硬塞進兩端）？
4. 有沒有「其實是屬性」的假實體（名額、截止時間）？有沒有「其實是實體」的假屬性（要記歷史的價格 → rate 表）？
5. 圖上每個框，題目情境裡都找得到對應的名詞嗎？（找不到＝可能過度設計）

## 1.7 應用系統七大建模模式（常見應用全覆蓋，對號入座）

| # | 模式 | 長相 | 誰會用到 |
|---|---|---|---|
| ① | **M:N＋屬性** | 關聯表＋`UNIQUE(fk1, fk2)` | 報名、訂單明細、選課、問卷填答 |
| ② | **時段／區間** | `start, end` 兩欄＋重疊判斷 | 場地／家教／門診預約、訂房 |
| ③ | **狀態機** | `status`＋`CHECK(IN (...))`＋程式管轉移 | 維修工單、訂單流程、候補隊伍 |
| ④ | **事件流（append-only）** | 紀錄表只插入不修改，狀態用查詢算 | 進場打卡、志工時數、病歷 |
| ⑤ | **快照 vs 即算** | 庫存存欄位（快、要維護）或 `SUM(進-出)`（慢、永遠對） | 庫存、名額、票券配額 |
| ⑥ | **價格隨時間** | `rate(對象, 起日, 價)` 查詢取當日價 | 季節房價、早鳥票種、時薪調整 |
| ⑦ | **階層／自參照** | `parent_id REFERENCES 自己` | 失物／商品分類樹 |

①在剛剛的報名系統已經完整示範（registration 表）。50 分鐘主線跑 ②③⑤；④⑥⑦保留為
**選讀／視進度**。七種都有可執行範例，你可依自己的題目替相關模式加星號，課後再查用。

In [ ]:
# 模式②「時段／區間」：重疊判斷 —— 兩區間重疊 ⟺ (A.start < B.end) AND (B.start < A.end)
con.executescript("""
DROP TABLE IF EXISTS booking;
CREATE TABLE booking(
  booking_id INTEGER PRIMARY KEY,
  room  TEXT NOT NULL,
  s     TEXT NOT NULL,   -- start
  e     TEXT NOT NULL,   -- end
  CHECK (s < e));
INSERT INTO booking(room, s, e) VALUES
  ('圓桌室', '2026-10-01 10:00', '2026-10-01 12:00'),
  ('圓桌室', '2026-10-01 14:00', '2026-10-01 16:00');
""")

def can_book(room, s, e):
    clash = con.execute("""SELECT COUNT(*) FROM booking
                           WHERE room = ? AND s < ? AND ? < e""", (room, e, s)).fetchone()[0]
    return "❌ 衝突，換時段" if clash else "✅ 可預約"

for s, e in [("2026-10-01 11:00","2026-10-01 13:00"),   # 跟 10–12 重疊
             ("2026-10-01 12:00","2026-10-01 14:00"),   # 剛好接在 12 點，不重疊（邊界！）
             ("2026-10-01 09:00","2026-10-01 17:00")]:  # 整段罩住
    print(f"{s[11:]}–{e[11:]} → {can_book('圓桌室', s, e)}")
# 想想：邊界要不要算重疊（<= 還是 <）？——這是「跟業主確認」的需求問題，不是技術問題

In [ ]:
# 模式③「狀態機」：CHECK 管值域、轉移表管路徑——雙保險
con.executescript("""
DROP TABLE IF EXISTS repair_ticket;
CREATE TABLE repair_ticket(
  tid    INTEGER PRIMARY KEY,
  title  TEXT NOT NULL,
  status TEXT NOT NULL DEFAULT '新單'
         CHECK (status IN ('新單','處理中','完成','已確認')));
INSERT INTO repair_ticket(title) VALUES ('浴室燈不亮');
""")
ALLOWED = {"新單": {"處理中"}, "處理中": {"完成", "新單"}, "完成": {"已確認"}, "已確認": set()}

def set_status(tid, new_status):
    cur = con.execute("SELECT status FROM repair_ticket WHERE tid = ?", (tid,)).fetchone()[0]
    if new_status not in ALLOWED[cur]:
        return f"❌ {cur} → {new_status}：不合法的跳轉"
    con.execute("UPDATE repair_ticket SET status = ? WHERE tid = ?", (new_status, tid))
    con.commit()
    return f"✅ {cur} → {new_status}"

print(set_status(1, "處理中"))
print(set_status(1, "完成"))
print(set_status(1, "新單"))          # 完成的單想跳回新單 → 程式層擋
try:
    con.execute("UPDATE repair_ticket SET status = '已讀' WHERE tid = 1")   # 亂寫狀態 → 約束層擋
except sqlite3.IntegrityError as e:
    print("✅ CHECK 擋下未知狀態 →", e)
# 這格先聚焦狀態模型；「先讀後寫」還不耐多人競態。競態版要把舊狀態寫進單一 UPDATE 的 WHERE 並檢查 rowcount，見 U06 武器①。
# 維修工單類題目的 demo 亮點：值域交給 CHECK、路徑交給轉移表，競態再交給原子語句。

In [ ]:
# 模式③加碼：「候補隊伍」＝狀態機 × window——排隊順位不用存欄位，用報名時間現算
con.executescript("""
DROP TABLE IF EXISTS waitlist_demo;
CREATE TABLE waitlist_demo(
  reg_id INTEGER PRIMARY KEY, member TEXT NOT NULL, 
  status TEXT NOT NULL CHECK (status IN ('報名','候補','取消')),
  reg_time TEXT NOT NULL);
INSERT INTO waitlist_demo(member, status, reg_time) VALUES
  ('林佳蓉','報名','2026-10-01 09:00'), ('陳威廷','報名','2026-10-01 09:05'),
  ('張雅筑','候補','2026-10-01 09:11'), ('王大明','候補','2026-10-01 09:20'),
  ('李心怡','候補','2026-10-01 10:02'), ('黃冠宇','取消','2026-10-01 09:03');
""")
print(q("""SELECT member AS 候補者,
             ROW_NUMBER() OVER (ORDER BY reg_time) AS 順位
          FROM waitlist_demo WHERE status = '候補'""").to_string(index=False))
print("→ 順位不存欄位（存了就要在每次取消時維護——快照陷阱！）；「有人退出→候補第 1 名遞補」")
print("   的完整交易寫法是 U06 的主菜。報名／掛號類題目請把這格加星號。")

In [ ]:
# 【選讀／視進度】模式④「事件流」：只插入、不修改——「現在的狀態」用查詢算出來
con.executescript("""
DROP TABLE IF EXISTS entry_log;
CREATE TABLE entry_log(
  id   INTEGER PRIMARY KEY,
  who  TEXT NOT NULL,
  ts   TEXT NOT NULL,
  kind TEXT NOT NULL CHECK (kind IN ('in','out')));
INSERT INTO entry_log(who, ts, kind) VALUES
  ('S001','2026-10-01 09:00','in'),  ('S002','2026-10-01 09:10','in'),
  ('S001','2026-10-01 11:00','out'), ('S003','2026-10-01 11:20','in'),
  ('S002','2026-10-01 12:00','out'), ('S002','2026-10-01 14:00','in');
""")
print(q("""SELECT who,
             SUM(kind = 'in')  AS 進場次,
             SUM(kind = 'out') AS 出場次,
             CASE WHEN SUM(kind='in') > SUM(kind='out') THEN '🟢 在館' ELSE '⚪ 已離開' END AS 目前
          FROM entry_log GROUP BY who""").to_string(index=False))
print("目前在館人數：", con.execute("SELECT SUM(kind='in') - SUM(kind='out') FROM entry_log").fetchone()[0])
print("→ 「在不在館」不存欄位：事件不可竄改（審計友善）、狀態永遠可重算。打卡、時數登錄類題目就是這招。")

In [ ]:
# 【選讀／視進度】模式④加碼：事件流 → 統計量——in/out 配對算「停留時數」（LAG 配 julianday）
print(q("""WITH t AS (
             SELECT who, ts, kind,
                    LAG(ts)   OVER (PARTITION BY who ORDER BY ts) AS prev_ts,
                    LAG(kind) OVER (PARTITION BY who ORDER BY ts) AS prev_kind
             FROM entry_log)
          SELECT who,
                 ROUND(SUM(CASE WHEN kind='out' AND prev_kind='in'
                           THEN (julianday(ts) - julianday(prev_ts)) * 24 END), 1) AS 累計時數
          FROM t GROUP BY who""").to_string(index=False))
print("→ S002 進了兩次、出了一次：未配對的 in 不計（還在館內）。志工時數、工時結算類題目：")
print("   存「事件」，時數永遠是「算出來的」——要對帳、要重算、要抓漏刷卡都做得到。")

In [ ]:
# 模式⑤「快照 vs 即算」：庫存欄位快、但忘了維護就漂移——用「對帳查詢」抓出來
con.executescript("""
DROP TABLE IF EXISTS stock_moves; DROP TABLE IF EXISTS products_s;
CREATE TABLE products_s(pid INTEGER PRIMARY KEY, pname TEXT NOT NULL,
                        stock_snapshot INTEGER NOT NULL DEFAULT 0);
CREATE TABLE stock_moves(id INTEGER PRIMARY KEY, pid INTEGER NOT NULL REFERENCES products_s(pid),
                         delta INTEGER NOT NULL, note TEXT);
INSERT INTO products_s(pid, pname, stock_snapshot) VALUES (1, '原子筆', 0), (2, '筆記本', 0);
""")
def move_stock(pid, delta, note, update_snapshot=True):
    con.execute("INSERT INTO stock_moves(pid, delta, note) VALUES (?,?,?)", (pid, delta, note))
    if update_snapshot:
        con.execute("UPDATE products_s SET stock_snapshot = stock_snapshot + ? WHERE pid = ?", (delta, pid))
    con.commit()

move_stock(1, +50, "進貨"); move_stock(1, -3, "賣出"); move_stock(2, +20, "進貨")
move_stock(1, -5, "賣出", update_snapshot=False)     # ← 某段新功能忘了維護快照（真實世界天天發生）
print(q("""SELECT p.pname, p.stock_snapshot AS 快照,
             COALESCE(SUM(m.delta), 0) AS 即算,
             CASE WHEN p.stock_snapshot = COALESCE(SUM(m.delta),0) THEN 'OK' ELSE '⚠️ 漂移！' END AS 對帳
          FROM products_s p LEFT JOIN stock_moves m ON p.pid = m.pid
          GROUP BY p.pid""").to_string(index=False))
print("→ 心法：**預設用即算**（永遠對）；量大才存快照，而且要「交易維護＋定期對帳」（U06 交易上場）。")

> **模式⑤是報告 Q&A 常考題**：存「剩餘名額」欄位（快照）要在每次報名／取消同步維護，忘一處就錯；
> 用查詢即算（如報名系統的剩餘名額查詢）永遠正確但每次要算。**預設用即算，量大再快照**——
> 而且快照必須配交易（U06 教）＋上面那格的對帳查詢。你的題目裡名額／庫存／餘額都是這一題。

快照 vs 即算五問（工作坊自問）：①這個數字被查的頻率？②算它要掃幾列？③誰負責維護快照、漏了會怎樣？
④能接受「稍舊」嗎？⑤有沒有對帳查詢當安全網？——五題答完，選擇自明。

In [ ]:
# 【選讀／視進度】模式⑥「價格隨時間」：rate(對象, 生效日, 價格)——「當日價」= 生效日 ≤ 當日的最新一筆
con.executescript("""
DROP TABLE IF EXISTS rate;
CREATE TABLE rate(room_type TEXT, start_date TEXT, price INTEGER,
                  PRIMARY KEY (room_type, start_date));
INSERT INTO rate VALUES
  ('雙人房','2026-01-01',2000), ('雙人房','2026-06-01',2500),
  ('雙人房','2026-12-30',4000),                       -- 跨年漲價！
  ('四人房','2026-01-01',3200);
""")
for d in ["2026-05-31", "2026-06-01", "2026-12-31"]:
    p = con.execute("""SELECT price FROM rate
                       WHERE room_type = '雙人房' AND start_date <= ?
                       ORDER BY start_date DESC LIMIT 1""", (d,)).fetchone()[0]
    print(f"雙人房 {d} 的房價 → {p}")
# 訂 12/29 起三晚的總價：日曆（遞迴 CTE，上個單元學的）× 逐日取價
q("""WITH RECURSIVE nights(d) AS (
        SELECT '2026-12-29' UNION ALL SELECT date(d,'+1 day') FROM nights WHERE d < '2026-12-31')
     SELECT nights.d AS 夜晚,
            (SELECT price FROM rate r WHERE r.room_type='雙人房' AND r.start_date <= nights.d
             ORDER BY r.start_date DESC LIMIT 1) AS 房價
     FROM nights""")
# 期望：2500 + 4000 + 4000 = 10500。改價格「插新列」而不是 UPDATE——歷史訂單的價格證據都還在

In [ ]:
# 【選讀／視進度】模式⑦「階層／自參照」：parent_id 指向自己 ＋ 遞迴 CTE 展開（上個單元的絕招在這用上）
con.executescript("""
DROP TABLE IF EXISTS category;
CREATE TABLE category(
  id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  parent_id INTEGER REFERENCES category(id));   -- 指向「自己這張表」；根節點是 NULL
INSERT INTO category(id, name, parent_id) VALUES
  (1,'全部商品',NULL),
    (2,'食品',1), (3,'文具',1),
      (4,'飲料',2), (5,'零食',2), (6,'筆類',3), (7,'紙品',3), (8,'手搖杯',4);
""")
print(q("""WITH RECURSIVE tree(id, name, depth, path) AS (
             SELECT id, name, 0, name FROM category WHERE parent_id IS NULL
             UNION ALL
             SELECT c.id, c.name, tree.depth + 1, tree.path || ' > ' || c.name
             FROM category c JOIN tree ON c.parent_id = tree.id)
           SELECT printf('%.*c%s', depth * 2, ' ', name) AS 目錄, path AS 路徑 FROM tree
           ORDER BY path""").to_string(index=False))
print("→ 失物分類、商品分類就這樣長；「某分類（含子孫）的總銷量」＝遞迴撈出子孫再 join。")

### 模式選用小結（工作坊前的最後裝備）

- 打開你的題目，把「必備功能」逐條對到模式表：**每一條功能 ≈ 一個模式 ＋ 一兩條約束**。
- 對不上的（例如選課分發演算法、問卷的卡方檢定）通常是「應用邏輯」——資料層只要把資料存對，演算法是 Python 的事。
- 判斷不了的（可多供應商嗎？邊界算重疊嗎？）記下來——**那是要問業主的需求問題**，你的報告裡「設計決策說明」就寫這些。

### 第 1 節 60 秒複盤

```
訪談逐字稿 ──名詞──► 實體（＋弱實體）
           ──動詞──► 關係 ──兩問一補──► 基數（1:1／1:N／M:N）＋參與（可為 0？）
ER 圖 ──三規則──► DDL ──七大模式──► 約束與查詢的慣用寫法
```

拿到任何題目（不只課堂這些例子），這條生產線就是你的 SOP。第 2 節回答另一個問題：**怎麼確認拆出來的表「沒拆錯」**——理論名字叫正規化。

## 1.8【選讀長案例・不占 135 分鐘主線】第三個完整病例：醫院掛號

### 需求 → ER／cardinality → 關聯綱要

> 「醫院分成多個**科別**，每位**醫師**隸屬一個科別並開出多個**門診場次**；每場有開始時間與名額。
> **病人**可以掛不同場次，同一場不能重複掛號；掛號取得看診序號，取消後仍要留下紀錄。」

先逐句問基數與參與：

- `department 1:N doctor`：一科多醫師；本案假設一位醫師只隸屬一科，所以 FK 放在 doctor。
- `doctor 1:N clinic_session`：一位醫師開多場；一場恰屬一位醫師，且 `(doctor_id, starts_at)` 不可重複。
- `patient M:N clinic_session`：病人可掛多場，一場可有多人；用 **appointment 關聯表**承接序號、狀態、掛號時間。
- 病人可以尚未掛號、剛開的場次可以 0 人；兩者都是 optional participation，報表要記得 LEFT JOIN。

```
department 1 ───< doctor 1 ───< clinic_session >───< appointment >─── 1 patient
                                      quota            visit_no
                                                       status, booked_at
```

轉成五張表：`department`、`doctor`、`patient`、`clinic_session`、`appointment`。這只是把模式① M:N 關聯表與模式⑤名額即算套進新情境，**不是新增第八種模式**。

In [ ]:
# 選讀病例：醫院掛號的完整 SQLite DDL（獨立記憶體資料庫，不影響其他示範）
hospital = sqlite3.connect(":memory:")
hospital.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE department(
  department_id INTEGER PRIMARY KEY,
  name          TEXT NOT NULL UNIQUE);

CREATE TABLE doctor(
  doctor_id     INTEGER PRIMARY KEY,
  department_id INTEGER NOT NULL REFERENCES department(department_id),
  name          TEXT NOT NULL,
  UNIQUE (department_id, name));

CREATE TABLE patient(
  patient_id INTEGER PRIMARY KEY,
  card_no    TEXT NOT NULL UNIQUE,
  name       TEXT NOT NULL);

CREATE TABLE clinic_session(
  session_id INTEGER PRIMARY KEY,
  doctor_id  INTEGER NOT NULL REFERENCES doctor(doctor_id),
  starts_at  TEXT NOT NULL,
  quota      INTEGER NOT NULL CHECK (quota > 0),
  UNIQUE (doctor_id, starts_at));

CREATE TABLE appointment(
  appointment_id INTEGER PRIMARY KEY,
  patient_id     INTEGER NOT NULL REFERENCES patient(patient_id),
  session_id     INTEGER NOT NULL REFERENCES clinic_session(session_id),
  visit_no       INTEGER NOT NULL CHECK (visit_no > 0),
  status         TEXT NOT NULL DEFAULT '已掛號'
                 CHECK (status IN ('已掛號','已報到','已看診','已取消')),
  booked_at      TEXT NOT NULL DEFAULT (datetime('now','+8 hours')),
  UNIQUE (patient_id, session_id),
  UNIQUE (session_id, visit_no));

INSERT INTO department VALUES (1,'家醫科'), (2,'眼科');
INSERT INTO doctor VALUES (1,1,'林醫師'), (2,2,'陳醫師');
INSERT INTO patient VALUES (1,'P0001','王怡君'), (2,'P0002','李明哲'), (3,'P0003','周雅雯');
INSERT INTO clinic_session VALUES
  (1,1,'2026-10-08 09:00',3), (2,1,'2026-10-08 14:00',2), (3,2,'2026-10-09 09:00',2);
INSERT INTO appointment(patient_id, session_id, visit_no, status) VALUES
  (1,1,1,'已掛號'), (2,1,2,'已報到'), (3,3,1,'已取消');
""")

hospital_query = """SELECT s.session_id, d.name AS doctor, s.starts_at, s.quota,
                           SUM(CASE WHEN a.status <> '已取消' THEN 1 ELSE 0 END) AS used_count,
                           s.quota - SUM(CASE WHEN a.status <> '已取消' THEN 1 ELSE 0 END) AS remaining
                    FROM clinic_session s JOIN doctor d ON s.doctor_id = d.doctor_id
                    LEFT JOIN appointment a ON s.session_id = a.session_id
                    GROUP BY s.session_id ORDER BY s.starts_at"""
print(pd.read_sql_query(hospital_query, hospital).to_string(index=False))

try:
    hospital.execute("INSERT INTO appointment(patient_id,session_id,visit_no) VALUES (1,1,3)")
except sqlite3.IntegrityError as err:
    print("✅ 同一病人同一場重複掛號被擋下 →", err)
hospital.rollback()

### 設計決策紀錄（病例收尾）

| 決策 | 為什麼 | 若需求改變 |
|---|---|---|
| `clinic_session` 獨立成表 | 「醫師」與「10/08 上午門診」不是同一件事；時間、名額描述場次 | 場次換診醫師只改該場 FK |
| `appointment` 不直接刪除取消資料 | `status='已取消'` 保留歷史，能算取消率；這是狀態，不是另一張表 | 若需完整轉移歷程，再加 append-only 狀態紀錄（模式④） |
| 剩餘名額不存欄位 | `quota − 有效掛號數` 可即算，避免取消後忘了加回 | 量大才做可重算快取，並配對帳 |
| 兩條複合 UNIQUE 分工 | 一條防病人重掛；另一條防同場發出兩個相同看診序號 | 若序號可重用，先把「何時可重用」問清楚再改 |

**刻意沒假裝 DDL 能做的事**：`CHECK` 看不到其他列，無法單獨保證「有效掛號數 ≤ quota」。正確做法是在 U06 用交易完成「查名額＋新增掛號」；本單元先把資料形狀與單列／唯一性約束設計對。

# 第 2 節：正規化——為什麼要拆表

## 2.1 反面教材：一張「什麼都記」的報名總表

不設計、直接開一張大表（想像這是從 Excel 匯入的），長這樣（欄位：學號 sid、姓名 sname、電話 phone、
活動編號 event_id、活動名稱 ename、活動日期 edate、社團編號 club_id、社團名稱 clname、社辦 office、報名時間 reg_time）：

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS reg_flat;
CREATE TABLE reg_flat(
  sid TEXT, sname TEXT, phone TEXT,
  event_id INTEGER, ename TEXT, edate TEXT,
  club_id INTEGER, clname TEXT, office TEXT,
  reg_time TEXT);
INSERT INTO reg_flat VALUES
 ('S001','林佳蓉','0912-111-111', 1,'Python 工作坊','2026-10-20', 1,'資料科學社','理學院 302','2026-09-30 10:00'),
 ('S002','陳威廷','0912-222-222', 1,'Python 工作坊','2026-10-20', 1,'資料科學社','理學院 302','2026-09-30 11:00'),
 ('S003','張雅筑','0912-333-333', 2,'SQL 快打旋風','2026-11-03', 1,'資料科學社','理學院 302','2026-10-01 09:00'),
 ('S001','林佳蓉','0912-111-111', 3,'行前會','2026-10-25', 2,'登山社','體育館 B1','2026-10-02 20:00');
""")
q("SELECT * FROM reg_flat")

In [ ]:
# 三種異常，現場出事：
print("── 更新異常 ──  資料科學社搬家，要改『每一列』；漏改一列就自相矛盾：")
con.execute("UPDATE reg_flat SET office='理學院 505' WHERE club_id=1 AND sid != 'S003'")  # 模擬漏改
print(q("SELECT DISTINCT clname, office FROM reg_flat WHERE club_id=1").to_string(index=False))
print("→ 同一個社團兩個社辦？資料庫自己都不知道哪個對。\n")

print("── 插入異常 ──  新社團「吉他社」還沒辦活動，就沒地方登錄（只能塞一堆 NULL 假列）")
print("── 刪除異常 ──  刪掉行前會唯一一筆報名：")
con.execute("DELETE FROM reg_flat WHERE event_id=3")
print("登山社還存在嗎？", q("SELECT COUNT(*) c FROM reg_flat WHERE club_id=2").iloc[0,0],
      "列 → 社團資訊跟著陪葬了")
con.commit()

## 2.2 病因診斷工具：函數相依（Functional Dependency）

**X → Y**：「知道 X 就唯一決定 Y」。統計語言：給定 X，Y 的條件分佈退化成單點。

⚠️ FD 來自**語意（業務規則）**，不是從目前的資料猜——資料只有 4 列時什麼巧合都可能成立（U01 那句：資料只能否證、不能證明）。

報名總表的 FD（從訪談語意讀出來；括號附中文對照）：

```
sid → sname, phone                     （學號 → 姓名、電話）
event_id → ename, edate, club_id       （活動 → 名稱、日期、所屬社團）
club_id → clname, office               （社團 → 名稱、社辦）
(sid, event_id) → reg_time             （誰＋哪場 → 報名時間）
```

**三種異常的病根都一樣**：非鍵屬性（office）依賴的不是整個主鍵，而是別的東西（club_id）——同一個事實被存了 N 次。

## 2.3 屬性閉包（closure）：機械化找出 candidate key

**X⁺ ＝ 從 X 出發，用 FD 能推出的所有屬性**。演算法就是「反覆套規則直到不動」——15 行 Python：

In [ ]:
def closure(attrs, fds):
    """attrs: 屬性集合；fds: [(左邊集合, 右邊集合), ...]；回傳 attrs 的閉包"""
    close = set(attrs)
    changed = True
    while changed:
        changed = False
        for lhs, rhs in fds:
            if set(lhs) <= close and not set(rhs) <= close:   # 左邊齊了、右邊還沒進來
                close |= set(rhs)
                changed = True
    return close

FDS = [({"sid"}, {"sname", "phone"}),
       ({"event_id"}, {"ename", "edate", "club_id"}),
       ({"club_id"}, {"clname", "office"}),
       ({"sid", "event_id"}, {"reg_time"})]
ATTRS = {"sid","sname","phone","event_id","ename","edate","club_id","clname","office","reg_time"}

for cand in [{"sid"}, {"event_id"}, {"sid", "event_id"}]:
    cl = closure(cand, FDS)
    print(f"{str(sorted(cand)):28s} 的閉包蓋住 {len(cl):2d}/{len(ATTRS)} 個屬性",
          "→ 是 key！" if cl == ATTRS else "")
# {sid, event_id} 推得出全部、其任何真子集都不行 → 它是（唯一的）candidate key

In [ ]:
# 幕後直播版：把 closure 的每一輪擴張印出來（考試手算就是走這個過程）
def closure_verbose(attrs, fds):
    close = set(attrs)
    print(f"起點：{sorted(close)}")
    rounds = 0
    changed = True
    while changed:
        changed = False; rounds += 1
        for lhs, rhs in fds:
            if set(lhs) <= close and not set(rhs) <= close:
                close |= set(rhs); changed = True
                print(f"  第 {rounds} 輪：套用 {sorted(lhs)} → {sorted(rhs)}，蓋到 {len(close)} 個屬性")
    return close

closure_verbose({"event_id"}, FDS)
print("→ event_id 推得出活動與社團的一切，但推不出「人」——所以它自己當不了 key。")

In [ ]:
# 換你：練習用 closure 檢查（改改看）
# Q：假設又冒出 FD「phone → sid」（一支電話只綁一人），{phone, event_id} 是 candidate key 嗎？
FDS2 = FDS + [({"phone"}, {"sid"})]
print("閉包 =", sorted(closure({"phone", "event_id"}, FDS2)))
print("蓋住全部？", closure({"phone", "event_id"}, FDS2) == ATTRS)
# 是的話：candidate key 可以不只一個；primary key 是你「選」的那一個

In [ ]:
# 加碼：把「找出所有 candidate key」也機械化——列舉子集＋closure＋極小性檢查
from itertools import combinations

def all_candidate_keys(attrs, fds):
    keys = []
    for r in range(1, len(attrs) + 1):
        for comb in combinations(sorted(attrs), r):
            s = set(comb)
            if any(k <= s for k in keys):        # 已包含更小的 key → 不極小，跳過
                continue
            if closure(s, fds) == set(attrs):
                keys.append(s)
    return keys

print("原 FD 的所有 candidate key：", all_candidate_keys(ATTRS, FDS))
print("加上 phone→sid 之後：      ", all_candidate_keys(ATTRS, FDS2))
print("再驗一個熟面孔：takes 的 key =",
      all_candidate_keys({"sid","cid","semester","grade"}, [({"sid","cid","semester"}, {"grade"})]))
# 2^10 = 1024 個子集全掃也只要一瞬間——考試手算閉包，實務交給程式；兩個都要會

In [ ]:
# FD 的「資料面照妖鏡」：groupby X 後數 Y 的 nunique——大於 1 就有反例（資料只能否證！）
flat = pd.read_sql_query("SELECT * FROM reg_flat", con)
for X, Y in [("club_id", "office"), ("sid", "sname"), ("event_id", "reg_time")]:
    n_bad = int((flat.groupby(X)[Y].nunique() > 1).sum())
    print(f"{X} → {Y}：{n_bad} 個反例", "💥" if n_bad else "（資料相容；語意仍需人來確認）")
print()
print("三行各有戲：① 2.1 的漏改當場現形（FD 應成立、資料髒了）；② 相容；")
print("③ 這條 FD 本來就不該成立（同活動兩人兩個報名時間）——資料幫你戳破錯的假設。")
print("拿到 AI 合成的資料，先對「你宣稱的 FD」跑一輪這面鏡子——資料品質檢查第一課。")

## 2.4 正規化等級：一關一關檢查

| 等級 | 要求（口語版） | 抓什麼病 |
|---|---|---|
| **1NF** | 每格都是原子值（不塞清單、不拆欄 tel1/tel2/tel3） | 「興趣：籃球,吉他,爬山」塞一格 |
| **2NF** | 非鍵屬性依賴**整個**主鍵，不能只依賴一部分 | `sid→sname`（只依賴複合鍵的一半） |
| **3NF** | 非鍵屬性不能**遞移**依賴（經過別的非鍵屬性） | `event_id→club_id→office` |
| **BCNF** | 每個 FD 的左邊都是 superkey | 3NF 的加強版（罕見的例外情境才分得出差） |

**實用判斷流程**：找出所有 FD → 用 closure 找 candidate key → 逐條 FD 檢查左邊是不是 key
→ 不是？把「左邊＋它決定的屬性」抽出去成新表，原表留左邊當 FK → 重複到每張表都乾淨。

**日常心法**（跟流程等價）：**一張表只講一個主題**；「這個欄位描述的是誰？」——描述的不是本表主鍵，就該搬家。

### 隨堂練習 D（1 分鐘口頭）：R(A, B, C)、FD 只有 A → B。key 是誰？這張表 BCNF 嗎？

<details><summary>答案</summary>
closure({A}) = {A,B} ≠ 全部 → A 不是 key；closure({A,C}) = 全部且極小 → key = {A,C}。
FD A→B 的左邊 A 不是 superkey → 違反 BCNF（也違反 2NF：B 只依賴 key 的一半）。拆成 (A,B) + (A,C)。
</details>

In [ ]:
# 先把 1NF 踩一遍：一格塞清單，報應立刻來
con.executescript("""
DROP TABLE IF EXISTS member_bad;
CREATE TABLE member_bad(name TEXT, skills TEXT);
INSERT INTO member_bad VALUES ('林佳蓉','吉他,烘焙'), ('陳威廷','電吉他'), ('張雅筑','攝影,吉他,排球');
""")
print("找「會吉他的人」只能 LIKE '%吉他%' →")
print(q("SELECT * FROM member_bad WHERE skills LIKE '%吉他%'").to_string(index=False))
print("→ 「電吉他」也命中（要不要算？規則說不清）；更致命的：")
print("   無法 GROUP BY 統計「每種才藝幾人」、無法用 UNIQUE 防「同人同才藝重複登錄」。")
print("   1NF 處方：拆 member(name) ＋ member_skill(name, skill) 一列一才藝——清單變成「可聚合、可約束」的資料。")

### 隨堂練習 F2（2 分鐘口頭）：三種「假 1NF」，各怎麼修？

1. `member(name, tel1, tel2, tel3)`——電話拆三欄
2. `event(title, tags)`，tags 存 `"迎新,免費,晚上"`
3. `survey(rid, answers_json)`，把 20 題答案整包塞一格 JSON

<details><summary>答案</summary>
三個都是「重複群組」的變形：① 第 4 支電話就崩＋查「誰有這支電話」要 OR 三欄 → `member_phone(member_id, phone)` 一列一支；② 逗號清單不可聚合不可約束 → `event_tag(event_id, tag)`；③ JSON 一格＝資料庫看不見的黑箱（不能索引、不能約束、統計要先解包）→ 一題一答一列（response 表）。
JSON 欄位不是罪——**當它只是「附加雜項」時可以**；但要查、要統計、要約束的資料就該是欄位。
</details>

## 2.5 動手分解：報名總表 → 3NF

逐條處理「左邊不是 key」的 FD：

```
club_id → clname, office            ⇒ 抽出 club(club_id, clname, office)
event_id → ename, edate, club_id    ⇒ 抽出 event(event_id, ename, edate, club_id→club)
sid → sname, phone                  ⇒ 抽出 member(sid, sname, phone)
剩下 registration(sid, event_id, reg_time)，主鍵 = (sid, event_id)
```

**這正是 1.5 節從 ER 推出來的報名系統 schema**——ER 設計得好，天生就接近 3NF；正規化是「驗算」，不是重來。

分解不能亂拆——必須**無損（lossless join）**：join 回來要跟原表一模一樣、不多不少。下一格用 SQL 驗證：

In [ ]:
# 無損分解驗證：拆出去的表 join 回來 ≟ 原始表（EXCEPT 雙向都空 = 完全一致）
con.executescript("""
DROP TABLE IF EXISTS d_member; DROP TABLE IF EXISTS d_event;
DROP TABLE IF EXISTS d_club;   DROP TABLE IF EXISTS d_reg;
DROP TABLE IF EXISTS flat0;
CREATE TABLE flat0(sid, sname, phone, event_id, ename, edate, club_id, clname, office, reg_time);
INSERT INTO flat0 VALUES
 ('S001','林佳蓉','0912-111-111',1,'Python 工作坊','2026-10-20',1,'資料科學社','理學院 302','2026-09-30 10:00'),
 ('S002','陳威廷','0912-222-222',1,'Python 工作坊','2026-10-20',1,'資料科學社','理學院 302','2026-09-30 11:00'),
 ('S003','張雅筑','0912-333-333',2,'SQL 快打旋風','2026-11-03',1,'資料科學社','理學院 302','2026-10-01 09:00'),
 ('S001','林佳蓉','0912-111-111',3,'行前會','2026-10-25',2,'登山社','體育館 B1','2026-10-02 20:00');
CREATE TABLE d_member AS SELECT DISTINCT sid, sname, phone FROM flat0;
CREATE TABLE d_event  AS SELECT DISTINCT event_id, ename, edate, club_id FROM flat0;
CREATE TABLE d_club   AS SELECT DISTINCT club_id, clname, office FROM flat0;
CREATE TABLE d_reg    AS SELECT DISTINCT sid, event_id, reg_time FROM flat0;
""")
recombine = """SELECT m.sid, m.sname, m.phone, e.event_id, e.ename, e.edate,
                      c.club_id, c.clname, c.office, r.reg_time
               FROM d_reg r JOIN d_member m ON r.sid = m.sid
               JOIN d_event e ON r.event_id = e.event_id
               JOIN d_club  c ON e.club_id = c.club_id"""
diff1 = len(q(f"SELECT * FROM flat0 EXCEPT {recombine}"))
diff2 = len(q(f"{recombine} EXCEPT SELECT * FROM flat0"))
print(f"原表 − 重組 = {diff1} 列；重組 − 原表 = {diff2} 列")
print("✅ 無損分解：資訊一滴不漏，但每個事實只存一次" if diff1 == diff2 == 0 else "❌ 拆錯了")
print("順帶一提：4 列 ×10 欄 = 40 格 → 拆完",
      sum(q(f"SELECT COUNT(*) c FROM {t}").iloc[0,0] for t in ["d_member","d_event","d_club","d_reg"]),
      "列（重複消失了）")

### 反例：亂拆會「生出鬼列」（lossy join）

無損不是自動的。把資料照「欄位順眼」亂拆成 `(sid, sname)` 和 `(sname, event_id)`——用**姓名**當接點。
兩位同名同姓的人一出現，join 就開始說謊：

In [ ]:
# 兩位「陳小美」：用姓名當橋拆表——join 回來多出不存在的報名！
con.executescript("""
DROP TABLE IF EXISTS L1; DROP TABLE IF EXISTS L2; DROP TABLE IF EXISTS mini_flat;
CREATE TABLE mini_flat(sid TEXT, sname TEXT, event_id INTEGER);
INSERT INTO mini_flat VALUES ('S001','陳小美',1), ('S099','陳小美',2), ('S002','王大明',1);
CREATE TABLE L1 AS SELECT DISTINCT sid, sname FROM mini_flat;
CREATE TABLE L2 AS SELECT DISTINCT sname, event_id FROM mini_flat;
""")
ghost = q("SELECT L1.sid, L1.sname, L2.event_id FROM L1 JOIN L2 ON L1.sname = L2.sname ORDER BY sid, event_id")
print(f"原表 3 列 → join 回來 {len(ghost)} 列：")
print(ghost.to_string(index=False))
print("\n→ S001 陳小美「被報名」了活動 2（其實是另一位陳小美）——資訊憑空出生＝有損（lossy）。")
print("   無損的實用判準：**接點必須是其中一張表的 key**（2.5 的接點 sid/event_id/club_id 都是）。")

### 隨堂練習 E（2 分鐘口頭）：無損？有損？

`R(sid, dept, dept_office)`，FD：`sid→dept`、`dept→dept_office`。三種拆法判斷：
1. `(sid, dept)` ＋ `(dept, dept_office)`
2. `(sid, dept_office)` ＋ `(dept, dept_office)`
3. `(sid, dept)` ＋ `(sid, dept_office)`

<details><summary>答案</summary>
1. ✅ 無損且保相依（接點 dept 是右表的 key）——標準答案。
2. ❌ 接點 dept_office 不是任一表的 key（兩系可共用同一辦公室）→ 可能生鬼列。
3. ✅ 無損（接點 sid 是兩表的 key）但 `dept→dept_office` 橫跨兩表驗不動——無損 ≠ 保相依（2.7 細講）。
</details>

## 2.6 第二個完整病例：訂單大表（含 2NF 違規）

電商匯出的「訂單明細總表」（你實習時真的會拿到這種檔）：

```
order_flat(order_id, prod_id, odate, cust_id, cust_name, city, prod_name, list_price, qty)
主鍵 =（order_id, prod_id）
FD：
  order_id → odate, cust_id            ← 只依賴主鍵的「一半」＝ 2NF 違規！
  cust_id → cust_name, city            ← 遞移 ＝ 3NF 違規
  prod_id → prod_name, list_price      ← 又是 2NF 違規
  (order_id, prod_id) → qty            ← 唯一健康的 FD
```

診斷 → 處方：**逐條抽表**（跟 2.5 同一招）：`orders(order_id, odate, cust_id)`、`customers(cust_id, cust_name, city)`、`products(prod_id, prod_name, list_price)`、`order_item(order_id, prod_id, qty)`——上個單元玩的 sales.db 就是這樣來的。下一格讓演算法自己走一遍：

In [ ]:
# 練習 E 驗證工作區：不服氣就用 SQL 驗——把「拆法 2」的鬼列抓出來（跑完再往下讀 2.6）
ck = sqlite3.connect(":memory:")
ck.executescript("""
CREATE TABLE R0(sid TEXT, dept TEXT, dept_office TEXT);
INSERT INTO R0 VALUES ('S001','統計','理學院 3F'), ('S002','數學','理學院 3F'), ('S003','統計','理學院 3F');
CREATE TABLE P1 AS SELECT DISTINCT sid, dept_office FROM R0;      -- 拆法 2：接點 dept_office
CREATE TABLE P2 AS SELECT DISTINCT dept, dept_office FROM R0;
""")
back = pd.read_sql_query("""SELECT P1.sid, P2.dept, P1.dept_office
                            FROM P1 JOIN P2 ON P1.dept_office = P2.dept_office
                            ORDER BY sid, dept""", ck)
print(f"原表 3 列 → join 回來 {len(back)} 列：")
print(back.to_string(index=False))
print("→ S001 同時屬於統計「和」數學系？兩系共用辦公室、接點不是 key → 鬼列誕生（判準應驗）。")

In [ ]:
# 用 closure 把「訂單大表」的診斷自動化：找 key → 逐條 FD 驗左邊 → 標出違規
ORDER_ATTRS = {"order_id","prod_id","odate","cust_id","cust_name","city","prod_name","list_price","qty"}
ORDER_FDS = [({"order_id"}, {"odate", "cust_id"}),
             ({"cust_id"}, {"cust_name", "city"}),
             ({"prod_id"}, {"prod_name", "list_price"}),
             ({"order_id", "prod_id"}, {"qty"})]

print("candidate key：", all_candidate_keys(ORDER_ATTRS, ORDER_FDS), "\n")
for lhs, rhs in ORDER_FDS:
    is_key = closure(lhs, ORDER_FDS) == ORDER_ATTRS        # 左邊的閉包蓋住全部 ⇔ 左邊是 superkey
    print(f"{'✅' if is_key else '💥'} {sorted(lhs)} → {sorted(rhs)}",
          "" if is_key else "　左邊不是 superkey → 把（左邊＋右邊）抽成新表")
print("\n抽完的四張表就是上個單元的 sales.db。注意 list_price（目前定價）在 products；")
print("order_item 若要記「成交當下的單價」要另存一欄——那不是重複，是不同的事實（見 2.9）。")

In [ ]:
# 訂單大表的異常也實際演一次：商品改名，部分依賴讓你「改到漏」
con.executescript("""
DROP TABLE IF EXISTS order_flat;
CREATE TABLE order_flat(order_id INTEGER, prod_id INTEGER, prod_name TEXT, qty INTEGER,
                        PRIMARY KEY (order_id, prod_id));
INSERT INTO order_flat VALUES
  (101, 7, '珍珠奶茶', 2), (102, 7, '珍珠奶茶', 1), (103, 7, '珍珠奶茶', 3), (103, 8, '烏龍綠', 1);
""")
con.execute("UPDATE order_flat SET prod_name = '珍珠鮮奶茶' WHERE order_id = 101 AND prod_id = 7")  # 漏改其他列！
con.commit()
print(q("SELECT DISTINCT prod_id, prod_name FROM order_flat WHERE prod_id = 7").to_string(index=False))
print("→ 同一個商品編號兩個名字——prod_name 只依賴主鍵的一半（prod_id），這就是 2NF 違規的實際代價。")
print("  拆出 products(prod_id, prod_name) 之後：改名＝改一列，全站同步。")

## 2.7【進階選讀／教師示範・不占 135 分鐘主線】把分解交給演算法

手動拆表會了，現在看**教科書的兩個標準演算法**（它們保證性質，你的手感負責品味）：

| 演算法 | 想法 | 保證 |
|---|---|---|
| **BCNF 分解**（decompose） | 找到違規 FD `X→Y` 就把表切成 `X⁺` 和 `R−(X⁺−X)`，遞迴到乾淨 | 無損 ✅・保相依 ❌（可能犧牲） |
| **3NF 合成**（synthesis） | 先把 FD 化成**最小覆蓋**，每個「同左邊」的 FD 群自成一表，沒 key 補 key 表 | 無損 ✅・保相依 ✅（但可能留一點冗餘） |

「保相依」＝分解後每條 FD 仍能在**單一表**內驗證。BCNF 有時做不到——經典反例：`(student, course) → prof`、`prof → course`（每位老師只教一門課）。下兩格把兩個演算法都跑起來：

In [ ]:
# BCNF 分解演算法（教學版：FD 投影用 closure 暴力算，小 schema 綽綽有餘）
from itertools import combinations

def project_fds(R, fds):
    """把 FD 集投影到屬性子集 R 上（列舉 R 的子集、用 closure 算它能決定什麼）"""
    R = set(R); out = []
    for r in range(1, len(R)):
        for lhs in combinations(sorted(R), r):
            rhs = (closure(set(lhs), fds) & R) - set(lhs)
            if rhs:
                out.append((set(lhs), rhs))
    return out

def bcnf(R, fds, depth=0):
    R = set(R)
    for lhs, rhs in project_fds(R, fds):
        if not (closure(lhs, fds) >= R):                     # lhs 不是 superkey → 違規！
            X = closure(lhs, fds) & R
            R1, R2 = X, R - (X - set(lhs))
            print("  " * depth + f"💥 {sorted(R)}")
            print("  " * depth + f"   違規 {sorted(lhs)} → {sorted(rhs)}，切成 {sorted(R1)} ＋ {sorted(R2)}")
            return bcnf(R1, fds, depth + 1) + bcnf(R2, fds, depth + 1)
    print("  " * depth + f"✅ {sorted(R)} 已是 BCNF")
    return [set(R)]

print("=== 報名總表的 BCNF 分解 ===")
result = bcnf(ATTRS, FDS)
print(f"\n共 {len(result)} 張表——跟 2.5 手拆的一模一樣！")

In [ ]:
# BCNF 的代價：經典反例——「每位老師只教一門課」
FDT = [({"student", "course"}, {"prof"}),      # 一個學生在一門課只有一位老師
       ({"prof"}, {"course"})]                 # 每位老師只教一門課（左邊不是 key！）
print("=== (student, course, prof) 的 BCNF 分解 ===")
r = bcnf({"student", "course", "prof"}, FDT)
print("\n注意：分解成（prof, course）＋（student, prof）之後，")
print("「(student,course)→prof」這條規則**橫跨兩張表**，單表驗不動了——這叫犧牲了相依保持。")
print("實務判決：這種罕見結構通常停在 3NF（保留一點冗餘、換取規則可驗證），並記錄設計決策。")

In [ ]:
# 「犧牲相依保持」是什麼感覺？拆完的兩張表各自守法，跨表的規則卻沒人管
con.executescript("""
DROP TABLE IF EXISTS student_prof; DROP TABLE IF EXISTS prof_course;
CREATE TABLE prof_course(prof TEXT PRIMARY KEY, course TEXT NOT NULL);   -- prof→course ✅ 單表可驗
CREATE TABLE student_prof(student TEXT, prof TEXT, PRIMARY KEY (student, prof));
INSERT INTO prof_course VALUES ('王老師','統計學'), ('李老師','統計學');   -- 兩位老師都教統計學
INSERT INTO student_prof VALUES ('陳同學','王老師'), ('陳同學','李老師');  -- 兩表都沒喊犯規！
""")
con.commit()
print(q("""SELECT s.student, s.prof, t.course
          FROM student_prof s JOIN prof_course t ON s.prof = t.prof""").to_string(index=False))
print("→ join 起來才看見：陳同學在「統計學」有兩位老師——(student,course)→prof 被違反了，")
print("   但這條規則橫跨兩表，任何單表約束都寫不出來。這就是 BCNF 可能付出的代價（實證版）。")

In [ ]:
# 3NF 合成演算法：最小覆蓋 → 同左邊成表 → 補 key 表
def minimal_cover(fds):
    G = [(set(l), {a}) for l, r in fds for a in r]            # ① 右邊拆成單屬性
    changed = True
    while changed:                                            # ② 刪左邊多餘屬性
        changed = False
        for i, (L, R) in enumerate(G):
            if len(L) > 1:
                for a in sorted(L):
                    if R <= closure(L - {a}, G):
                        G[i] = (L - {a}, R); changed = True; break
    i = 0
    while i < len(G):                                         # ③ 刪多餘 FD
        H = G[:i] + G[i+1:]
        if G[i][1] <= closure(G[i][0], H):
            G = H
        else:
            i += 1
    return G

def synth_3nf(attrs, fds):
    G = minimal_cover(fds)
    groups = {}
    for L, R in G:                                            # ④ 同左邊的 FD 併成一表
        groups.setdefault(frozenset(L), set()).update(R)
    tables = [set(L) | R for L, R in groups.items()]
    keys = all_candidate_keys(attrs, fds)                     # ⑤ 沒表含 key → 補一張 key 表
    if not any(any(set(k) <= t for t in tables) for k in keys):
        tables.append(set(next(iter(keys))))
    return [t for t in tables if not any(t < u for u in tables)]   # 去被包含的表

print("報名總表的 3NF 合成：")
for t in synth_3nf(ATTRS, FDS):
    print("  ", sorted(t))
print("\n訂單大表的 3NF 合成（第二病例也交給演算法對答案）：")
for t in synth_3nf(ORDER_ATTRS, ORDER_FDS):
    print("  ", sorted(t))
print("\n→ 兩案都與手拆殊途同歸，而且這條路**保證**無損＋保相依。")
print("   AI 給你 schema 時，把 FD 列出來跑這兩格——它拆得對不對，演算法說了算。")

In [ ]:
# 回頭單獨看「最小覆蓋」做了什麼：把 FD 集合瘦身到「一條都不能少」
demo_fds = [({"A"}, {"B"}), ({"B"}, {"C"}), ({"A"}, {"C"}),      # A→C 可由 A→B→C 推出（冗餘）
            ({"A", "B"}, {"D"})]                                  # 左邊的 B 多餘（A 就決定 B）
def show(fds): return "、".join(f"{''.join(sorted(l))}→{''.join(sorted(r))}" for l, r in fds)
print("原始 FD：", show(demo_fds))
print("最小覆蓋：", show(minimal_cover(demo_fds)))
print("→ 冗餘的 A→C 被刪、AB→D 瘦成 A→D。合成前先瘦身，拆出來的表才不會多長冗餘欄。")

### 選讀練習 G：key 不只一把的經典題

`R(A, B, C, D)`，FD：`AB → C`、`C → D`、`D → A`。所有 candidate key 是？（提示：交給程式，然後**手算驗證其中一把**）

<details><summary>答案</summary>
三把：{A,B}、{B,C}、{B,D}——B 出現在每一把裡（B 不在任何 FD 右邊 → 任何 key 必含 B）。
「D→A」讓 A 不再稀有；環狀 FD（C→D→A→…間接回 C 的左邊）常製造多把 key。BCNF 判斷時「左邊是不是 key」要對**每一把** key 檢查。
</details>

In [ ]:
# 練習 G 工作區：改這兩行，讓演算法告訴你答案
ex_attrs = {"A", "B", "C", "D"}
ex_fds = [({"A", "B"}, {"C"}), ({"C"}, {"D"}), ({"D"}, {"A"})]
print("所有 candidate key：", all_candidate_keys(ex_attrs, ex_fds))
# 手算驗證 {B, C}：closure({B,C}) = BC → +D（C→D）→ +A（D→A）= 全部 ✔

## 2.8【進階選讀／不占 135 分鐘主線】多值相依與 4NF

「王教授**會教**的課」和「王教授**寫過**的書」互相獨立，硬塞同一張表會怎樣？

In [ ]:
# MVD 病例：兩組互相獨立的多值，同表就得存「所有組合」
con.executescript("""
DROP TABLE IF EXISTS prof_info;
CREATE TABLE prof_info(prof TEXT, course TEXT, book TEXT);
INSERT INTO prof_info VALUES
 ('王教授','統計學','《迴歸不難》'), ('王教授','統計學','《ANOVA 圖解》'),
 ('王教授','機率論','《迴歸不難》'), ('王教授','機率論','《ANOVA 圖解》');
""")
print(q("SELECT * FROM prof_info").to_string(index=False))
n = q("SELECT COUNT(*) c FROM prof_info").iloc[0,0]
print(f"\n2 門課 × 2 本書 = {n} 列。再寫一本書 → 得補 2 列；漏補 → 資料自相矛盾。")
print("這叫多值相依（MVD）：prof ↠ course、prof ↠ book。**4NF 的處方：各自成表**——")

In [ ]:
# 4NF 處方箋實作：拆成兩表，資訊不減、組合爆炸消失
con.executescript("""
DROP TABLE IF EXISTS prof_courses; DROP TABLE IF EXISTS prof_books;
CREATE TABLE prof_courses AS SELECT DISTINCT prof, course FROM prof_info;
CREATE TABLE prof_books   AS SELECT DISTINCT prof, book   FROM prof_info;
""")
n1 = q("SELECT COUNT(*) c FROM prof_courses").iloc[0,0]
n2 = q("SELECT COUNT(*) c FROM prof_books").iloc[0,0]
n3 = len(q("""SELECT a.prof, a.course, b.book FROM prof_courses a JOIN prof_books b ON a.prof = b.prof
             EXCEPT SELECT prof, course, book FROM prof_info"""))
print(f"拆完：授課 {n1} 列 ＋ 著作 {n2} 列 = {n1+n2} 列（原本 4 列的組合表）")
print(f"join 回來 ＝ 原表？多出 {n3} 列 → {'✅ 無損' if n3 == 0 else '❌'}")
print("再寫一本新書：著作表加 1 列就好——不用再「每門課配一列」。")
print("你的專題聞到「兩組互相獨立的清單塞同一表」就拆——通常 ER 畫對根本不會發生。")

## 2.9 什麼時候「故意不正規化」？

正規化的代價是查詢要 join。兩個合法的反正規化時機：

1. **報表快取**：每晚把「各活動報名統計」算好存成一張表（讀多寫少、可重算就不怕錯）——1.7 模式⑤「快照」的合法應用。
2. **歷史快照**：訂單明細要存「**成交當下**的單價」——商品表的價格之後會變，這不是重複，是**不同的事實**！（交易類題目注意：`order_item.unit_price` 是對的設計，別「正規化」掉它；對照 1.7 模式⑥的 rate 表，那是「查價的規則」，這是「成交的證據」。）

### 正規化 FAQ（工作坊常見兩問）

- **「要 100% 正規化嗎？」**——OLTP 端以 3NF 為底線、BCNF 看情況（2.7 的代價）；上面兩種反正規化要**寫進設計決策**。
- **「兩三列的小代碼表（狀態、類別）也要拆嗎？」**——值域固定的小清單用 `CHECK(IN (...))` 就好；會長大、帶屬性（分類的說明、排序）才升格成表。

In [ ]:
# 合法反正規化之一「報表快取」：算好存起來、標上時間戳、隨時可重算
con.executescript("""
DROP TABLE IF EXISTS reg_stats;
CREATE TABLE reg_stats AS
SELECT e.event_id, e.title, COUNT(r.reg_id) AS n_reg, datetime('now','+8 hours') AS computed_at
FROM event e LEFT JOIN registration r ON e.event_id = r.event_id AND r.status = '報名'
GROUP BY e.event_id;
""")
con.commit()
print(q("SELECT * FROM reg_stats").to_string(index=False))
print("\n→ 儀表板讀這張快取表（快）；每晚重算一次。合法的理由：①真相仍在原表、隨時可重算")
print("   ②讀多寫少。快取沒更新叫「過期」，不叫「錯」——跟亂存重複資料是兩回事。")

### 2.9.1【選讀實測・不占 135 分鐘主線】JOIN 成本 vs 冗餘維護

同一份銷售資料做兩版，直接量兩種成本：

- **正規化版**：`sale_norm` 只存 `product_id`，查分類營收時 JOIN `product`。
- **反正規化版**：`sale_denorm` 每筆銷售重複存商品名稱與分類，查報表少一次 JOIN。

下格使用固定 seed、相同資料與獨立 `:memory:` 連線；結果不依賴前面的 `club.db`，也不污染後續。計時只比較**這次執行環境**，重點不是背毫秒數，而是同時看見「讀取少做一次 JOIN」與「改一個分類要維護很多份」的交換。

In [ ]:
# 固定資料、固定 seed：量正規化 JOIN 與反正規化掃描的讀取成本
import random, statistics, time

bench = sqlite3.connect(":memory:")
bench.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE product(
  product_id   INTEGER PRIMARY KEY,
  product_name TEXT NOT NULL UNIQUE,
  category     TEXT NOT NULL);
CREATE TABLE sale_norm(
  sale_id    INTEGER PRIMARY KEY,
  product_id INTEGER NOT NULL REFERENCES product(product_id),
  qty        INTEGER NOT NULL CHECK (qty > 0));
CREATE TABLE sale_denorm(
  sale_id      INTEGER PRIMARY KEY,
  product_id   INTEGER NOT NULL,
  product_name TEXT NOT NULL,
  category     TEXT NOT NULL,
  qty          INTEGER NOT NULL CHECK (qty > 0));
CREATE INDEX ix_sale_norm_product ON sale_norm(product_id);
CREATE INDEX ix_sale_denorm_product ON sale_denorm(product_id);
""")

product_count = 400
sale_count = 60000
bench_rng = random.Random(404)
product_rows = [(pid, f"Product-{pid:04d}", f"C{pid % 20:02d}")
                for pid in range(1, product_count + 1)]
product_map = {pid: (name, category) for pid, name, category in product_rows}
sale_rows = [(sale_id, bench_rng.randint(1, product_count), bench_rng.randint(1, 5))
             for sale_id in range(1, sale_count + 1)]
denorm_rows = [(sale_id, pid, product_map[pid][0], product_map[pid][1], qty)
               for sale_id, pid, qty in sale_rows]
bench.executemany("INSERT INTO product VALUES (?,?,?)", product_rows)
bench.executemany("INSERT INTO sale_norm VALUES (?,?,?)", sale_rows)
bench.executemany("INSERT INTO sale_denorm VALUES (?,?,?,?,?)", denorm_rows)
bench.commit()

norm_sql = """SELECT p.category, SUM(s.qty)
              FROM sale_norm s JOIN product p ON s.product_id = p.product_id
              GROUP BY p.category ORDER BY p.category"""
denorm_sql = """SELECT category, SUM(qty)
                FROM sale_denorm GROUP BY category ORDER BY category"""

def median_query_ms(conn, sql, repeats=9):
    list(conn.execute(sql))                         # warm-up，不把第一次載入算進去
    samples = []
    for _ in range(repeats):
        started = time.perf_counter()
        list(conn.execute(sql))
        samples.append((time.perf_counter() - started) * 1000)
    return statistics.median(samples)

norm_result = list(bench.execute(norm_sql))
denorm_result = list(bench.execute(denorm_sql))
assert norm_result == denorm_result                 # 先確定兩版回答相同問題
norm_ms = median_query_ms(bench, norm_sql)
denorm_ms = median_query_ms(bench, denorm_sql)
print(f"固定 {sale_count:,} 筆銷售；兩版結果相同：{norm_result == denorm_result}")
print(f"正規化 JOIN 中位數：{norm_ms:.2f} ms")
print(f"反正規化中位數：   {denorm_ms:.2f} ms")
print(f"本機這次讀取比值（JOIN / 冗餘）：{norm_ms / denorm_ms:.2f}x")

In [ ]:
# 同一個實驗接著量寫入面：商品換分類，真相是一列，冗餘副本卻散在許多列
target_id = 17
new_category = "C99"
copy_count = bench.execute(
    "SELECT COUNT(*) FROM sale_denorm WHERE product_id = ?", (target_id,)).fetchone()[0]

started = time.perf_counter()
norm_update = bench.execute(
    "UPDATE product SET category = ? WHERE product_id = ?", (new_category, target_id))
bench.commit()
norm_update_ms = (time.perf_counter() - started) * 1000

stale_count = bench.execute(
    "SELECT COUNT(*) FROM sale_denorm WHERE product_id = ? AND category <> ?",
    (target_id, new_category)).fetchone()[0]
norm_view = bench.execute(
    "SELECT category, SUM(qty) FROM sale_norm JOIN product USING(product_id) WHERE product_id = ?",
    (target_id,)).fetchone()
denorm_view = bench.execute(
    "SELECT category, SUM(qty) FROM sale_denorm WHERE product_id = ? GROUP BY category",
    (target_id,)).fetchone()

started = time.perf_counter()
denorm_update = bench.execute(
    "UPDATE sale_denorm SET category = ? WHERE product_id = ?", (new_category, target_id))
bench.commit()
denorm_update_ms = (time.perf_counter() - started) * 1000

remaining_mismatch = bench.execute("""SELECT COUNT(*)
    FROM sale_denorm sd JOIN product p USING(product_id)
    WHERE sd.category <> p.category""").fetchone()[0]
print(f"正規化：改 {norm_update.rowcount} 列，{norm_update_ms:.3f} ms；查詢立即看到 {norm_view}")
print(f"只改主檔時：反正規化仍有 {stale_count}/{copy_count} 份舊分類，查到 {denorm_view}")
print(f"補同步：改 {denorm_update.rowcount} 列，{denorm_update_ms:.3f} ms；不一致剩 {remaining_mismatch} 列")
print("→ 少一次 JOIN 可能省讀取時間；代價是寫入放大，而且每條寫入路徑都不能漏同步。")

**怎麼讀這個實驗**：毫秒與倍率會隨硬體、SQLite 版本、資料量、索引及快取改變；可重現的是成本形狀。

```
正規化：報表讀取 = 掃事實 + JOIN　　　　分類改名 = 更新 1 列
反正規化：報表讀取 = 掃一表　　　　　　分類改名 = 更新 N 份副本 + 防漏同步
```

所以不能用「JOIN 一定慢」當反正規化理由。先量真實查詢，再把三件事寫進設計決策：**哪個查詢慢、冗餘由誰更新、如何對帳／重建**。答不出來就維持 3NF；若只是報表需求，上一格那種「來源仍正規化、快取可重算」通常更安全。

### 給統計／BI 的一頁：星型 schema（OLAP 的世界觀）

分析型資料庫（data warehouse）刻意用「**一張大事實表**（fact：一筆交易一列）＋**幾張維度表**（dim：顧客/商品/日期）」的星型結構：

```
            dim_customer          dim_product
                  ▲                    ▲
                  └──── fact_sales ────┘
                          ▲
                      dim_date（日期也做成表！）
```

- **粒度（grain）先講清楚**：fact 一列是「一筆訂單」還是「一天一店一品」？粒度定了，報表才知道能問什麼。
- **dim_date 是本課所有招式的合體**：遞迴 CTE 生日曆＋strftime 重編碼，存成表之後每張報表 join 它。
- OLTP 求寫入正確 → 高度正規化；OLAP 求讀取方便 → 星型（維度表刻意反正規化）。**兩個世界，兩套哲學**（U08 對決）。
- 上個單元的 sales.db 就是一顆小星星：orders 是 fact，customers/products 是 dim。

下一格把「dim_date」真的建出來——你的專題報表可以直接抄這招：

In [ ]:
# 建 dim_date：遞迴生成 2026 全年日曆＋常用重編碼，存成表（之後所有報表 join 它）
con.executescript("""
DROP TABLE IF EXISTS dim_date;
CREATE TABLE dim_date AS
WITH RECURSIVE cal(d) AS (
    SELECT '2026-01-01'
    UNION ALL
    SELECT date(d, '+1 day') FROM cal WHERE d < '2026-12-31')
SELECT d,
       strftime('%Y-%m', d)             AS ym,
       CAST(strftime('%m', d) AS INT)   AS m,
       CAST(strftime('%w', d) AS INT)   AS wd,
       CASE WHEN strftime('%w', d) IN ('0','6') THEN 1 ELSE 0 END AS is_weekend,
       CASE CAST((strftime('%m', d) + 2) / 3 AS INT) WHEN 1 THEN 'Q1' WHEN 2 THEN 'Q2'
            WHEN 3 THEN 'Q3' ELSE 'Q4' END AS quarter
FROM cal;
""")
con.commit()
print(q("SELECT COUNT(*) AS 天數, SUM(is_weekend) AS 週末天數 FROM dim_date").to_string(index=False))
q("SELECT * FROM dim_date WHERE d BETWEEN '2026-02-26' AND '2026-03-02'")

In [ ]:
# 星型查詢初體驗：合成一張迷你 fact 表，join dim_date 做「週末 vs 平日」分析
import numpy as np
rng = np.random.default_rng(7)
days_df = pd.read_sql_query("SELECT d, is_weekend FROM dim_date", con)
sampled = days_df.sample(n=3000, replace=True, random_state=7).reset_index(drop=True)
fact = pd.DataFrame({
    "d": sampled.d,
    "amount": (rng.lognormal(4, 0.5, 3000) * (1 + 0.35 * sampled.is_weekend.to_numpy())).round(0)})
fact.to_sql("fact_sales", con, if_exists="replace", index=False)
q("""SELECT dd.quarter, dd.is_weekend, COUNT(*) AS n, ROUND(AVG(f.amount)) AS 平均客單
     FROM fact_sales f JOIN dim_date dd ON f.d = dd.d
     GROUP BY dd.quarter, dd.is_weekend ORDER BY dd.quarter, dd.is_weekend""")
# fact 只存「事實」（日期、金額）；時間的一切語意（季、週末⋯）都住在 dim_date——星型的分工

In [ ]:
# 星型的甜頭：換一個分析角度＝換一個 dim 欄位，fact 一個字不動
q("""SELECT dd.ym AS 月份, ROUND(SUM(f.amount)) AS 月營收,
        ROUND(AVG(CASE WHEN dd.is_weekend = 1 THEN f.amount END)) AS 週末平均單
     FROM fact_sales f JOIN dim_date dd ON f.d = dd.d
     GROUP BY dd.ym ORDER BY dd.ym LIMIT 6""")
# 想加「學期別」「連假旗標」？——只要在 dim_date 加欄位，全部報表自動獲得新視角

### 隨堂練習 H（3 分鐘，動手）：用星型回答一個新問題

「**Q3 的週末總營收**是多少？」——fact_sales 一個字不改，全靠 dim_date 的欄位。

<details><summary>參考解</summary>

```sql
SELECT ROUND(SUM(f.amount)) AS q3_weekend_rev
FROM fact_sales f JOIN dim_date dd ON f.d = dd.d
WHERE dd.quarter = 'Q3' AND dd.is_weekend = 1;
```
時間語意（季、週末）全部住在維度表——這就是「換角度不動 fact」的星型甜頭。
</details>

In [ ]:
# 練習 H 工作區
# TODO




## 2.10 隨堂練習 F：診斷這兩張表（4 分鐘）

**表一** `orders(order_id PK, cust_id, cust_name, cust_phone, odate)`
**表二** `ticket(ticket_no PK, showtime_id, seat_no, movie_title, starts_at)`，且 `showtime_id → movie_title, starts_at`

各違反第幾正規化？怎麼修？

<details><summary>答案</summary>

**表一**：3NF 違反（遞移：order_id→cust_id→cust_name/phone）。修：抽出 `customers(cust_id, name, phone)`，orders 留 FK。
**表二**：3NF 違反（ticket_no→showtime_id→movie_title/starts_at）。修：抽出 `showtime(showtime_id, movie_id→movie, starts_at)`；ticket 留 (showtime_id, seat_no) 並設 UNIQUE——這正是 1.6 寫過的場次×座位 schema。
</details>

## 2.11【AI 協作】讓 AI 當你的設計助手——與反向質詢

**第一輪（生成）**——把你題目的「情境」段落原文貼給 AI：

> 以下是一個資訊系統的需求描述：（貼上你的題目情境＋必備功能）。
> 請做：1) 名詞動詞分析找出實體與關係並標基數；2) 給出 3NF 的 SQLite DDL（含 PK/FK/CHECK/UNIQUE/DEFAULT）；
> 3) 說明每條 UNIQUE 與 CHECK 對應需求裡的哪句話。

**第二輪（反向質詢）**——AI 最有價值的用法不是「幫我寫」，而是「**幫我挑毛病**」：

> 這是我的 schema：（貼 DDL）。請找出三個潛在設計問題：
> 有沒有更新異常風險？有沒有該加而沒加的約束？「XX 情境」發生時這個 schema 撐得住嗎？

**驗收 SOP**：AI 給的 schema，跑本節的完整流程——塞資料、演異常、列 FD、跑 closure／合成演算法、檢查每張表「一表一主題」。**它給的每個 CHECK 你都要能說出對應需求的哪句話**，說不出的就刪掉或問到懂。

# 工作坊（35 分鐘）：設計你自己的專題 schema

**流程**（兩人一組互當「業主」，但各做各的題目）：

1. （5 分）對自己題目的情境做**名詞動詞分析**——填下面的模板
2. （10 分）畫 ER 草圖（紙筆即可）：實體、關係、基數、參與
3. （15 分）寫 DDL 草稿到工作區——先求骨架正確，示範資料之後的課堂實作再灌
4. （5 分）**互評**：交換螢幕，依下方 8 分 rubric 留下「一項證據充分＋一項優先修正」；再跑最下面的**自檢器**

## 分析模板（複製到你的筆記）

| 實體 | 主鍵 | 關鍵屬性 | 備註 |
|---|---|---|---|
| （例）event | event_id | title, quota, deadline | 名額>0 |

| 關係 | 基數 | 實作 | 自帶屬性 |
|---|---|---|---|
| （例）member 報名 event | M:N | registration 表 | status, reg_time |

| 我用到的建模模式（1.7 對照表） | 用在哪 |
|---|---|
| （例）狀態機 | registration.status |

### 常見應用「類型原型」起手式（防呆版；表可增減，最終以你的正規化分析為準）

| 類型 | 核心表（骨架） | 靈魂約束／模式 |
|---|---|---|
| 報名／選課／填答類 | 人・對象（活動/課程/問卷）・關聯表（狀態、時間） | UNIQUE(人, 對象)・狀態機③・名額⑤ |
| 預約／掛號／訂房類 | 資源（場地/醫師/房）・使用者・booking（起迄） | 區間②・CHECK(s<e)・UNIQUE(資源, 時段) |
| 交易流水類（點餐/售票/買賣） | 主檔・order・order_item（弱實體） | 明細存成交價・配額⑤・時序價格⑥ |
| 進銷存／庫存類 | product・supplier・進出單(_item) | 庫存⑤（快照＋對帳）・階層⑦（分類） |
| 工單／狀態流轉類 | 單據・處理人・指派紀錄 | 狀態機③・事件流④（處理歷程） |
| 出勤／時數／打卡類 | 人・活動・log（append-only） | 事件流④・M:N① |
| 目錄／比對類（失物/藏書） | 兩側刊登・分類樹・match | 階層⑦・狀態機③ |

用法：找到最像你題目的列 → 對照 1.6／1.7 的可跑範例 → 工作坊直接開寫。你的題目規格（指派時發給）若與此表有出入，以你的正規化分析為準。

### 命名慣例（全班統一，AI 也看得懂）

- 表名／欄名：**英文 snake_case**（`order_item`、`check_in`）；單複數擇一、**全案一致**。
- 主鍵：`表名_id`；FK 與被參照的主鍵**同名**（`booking.room_id → room.room_id`）——join 條件一眼即懂。
- 日期時間欄：`*_at`（時間點）、`*_date`（日期）；布林 `is_*`（存 0/1）。
- 避開保留字（`order` 是保留字！用 `orders`——sales.db 就是這樣躲的）。
- **識別字一律英文**（表名、欄名、變數）；中文放註解、字串資料與報表顯示別名（`AS 姓名`）——跨工具、跨 AI 最不容易出怪錯，也是業界慣例。

In [ ]:
# 工作坊 DDL 工作區：把你的 schema 草稿寫成真的（骨架照抄 1.5 的範例改）
my = sqlite3.connect("myproject.db")
my.executescript("""
PRAGMA foreign_keys = ON;

-- TODO: 你的表（≥4 張、PK/FK 齊、≥3 種其他約束）
-- CREATE TABLE ...;

""")
print("目前的表：", [r[0] for r in my.execute(
    "SELECT name FROM sqlite_master WHERE type='table'")])

In [ ]:
# 工作坊加碼工作區：把你題目的 FD 寫下來，讓今天的演算法幫你驗 schema
my_attrs = set()      # TODO：{"..."}
my_fds  = []          # TODO：[({"lhs"}, {"rhs1", "rhs2"}), ...]

if my_attrs and my_fds:
    print("candidate keys：", all_candidate_keys(my_attrs, my_fds))
    for t in synth_3nf(my_attrs, my_fds):
        print("  建議表：", sorted(t))
else:
    print("（填好 my_attrs / my_fds 再跑；格式照 2.6 的訂單病例）")

In [ ]:
# schema 自檢器：對任何 SQLite 連線做健檢（工作坊當場跑；之後每次改 schema 都跑）
def schema_health(conn):
    tables = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
    if not tables:
        print("（還沒有表）"); return
    fk_on = conn.execute("PRAGMA foreign_keys").fetchone()[0]
    print(f"PRAGMA foreign_keys = {'ON ✅' if fk_on else 'OFF ⚠️ 記得打開！'}\n")
    n_fk_total = 0
    for t in tables:
        cols = conn.execute(f"PRAGMA table_info('{t}')").fetchall()
        fks  = conn.execute(f"PRAGMA foreign_key_list('{t}')").fetchall()
        n_fk_total += len(fks)
        pk  = [c[1] for c in cols if c[5] > 0]
        nn  = sum(1 for c in cols if c[3])
        sql = conn.execute("SELECT sql FROM sqlite_master WHERE name = ?", (t,)).fetchone()[0].upper()
        flags = []
        if not pk: flags.append("⚠️ 沒有 PRIMARY KEY")
        if "CHECK"  in sql: flags.append("CHECK✓")
        if "UNIQUE" in sql: flags.append("UNIQUE✓")
        if "DEFAULT" in sql: flags.append("DEFAULT✓")
        print(f"{t:16s} 欄位 {len(cols):2d}・PK {pk or '無'}・FK {len(fks)}・NOT NULL {nn}  {' '.join(flags)}")
    print(f"\n合計 {len(tables)} 表、{n_fk_total} 條 FK",
          "→ 至少 4 表？" if len(tables) < 4 else "✅ 表數達標")

print("拿報名示範庫試跑：\n")
schema_health(con)
print("\n你的草稿：\n")
schema_health(my)

### 互評 rubric（8 分；只為修草稿，不是成績）

| 面向 | 2 分：有可指認證據 | 1 分：方向對但缺漏 | 0 分：尚未表達 |
|---|---|---|---|
| ER 與基數 | 每個實體有主鍵；每條關係雙向 cardinality／optional participation 都標清楚 | 圖已成形，但有一處基數或參與未交代 | 只有表名清單，或 ER 與 DDL 對不起來 |
| PK／FK 結構 | 1:N 的 FK 在 N 方；M:N 有關聯表；所有表都有 PK | 大致正確，但有幽靈 FK／缺一個 key | 用清單欄直連 M:N，或主要表無 PK |
| 業務約束 | 至少能指出 UNIQUE、CHECK、NOT NULL 各自對應哪句需求 | 有約束，但說不出一條的需求來源 | 規則只寫在註解／Python 想像中 |
| 正規化與決策 | 每表說得出一列粒度；冗餘／快照有理由與對帳法 | 疑似兩個主題混表，尚能指出待確認處 | 同一事實散存多處且沒有維護策略 |

互評者最後只留兩句：`證據充分：＿＿（指出圖／DDL）`、`優先修正：＿＿（給可執行改法）`。不要只寫「看起來不錯」。若未滿 6 分，先修 0 分項再加新功能。

**換位攻擊 5 連問（拿來找 rubric 的證據）**：

1. 「我**重複**送出同一筆（報名／訂位／填答），你的哪條 UNIQUE 擋我？」
2. 「我塞一個**不存在的編號**（幽靈 FK），會怎樣？`PRAGMA foreign_keys` 開了嗎？」
3. 「我把**狀態亂改**（完成→新單）、**數字改負**（名額 −3），哪一層擋？」
4. 「你這張表**一列代表什麼**？」（答不出來＝主題不清＝可能要拆）
5. 「這個欄位**描述的是誰**？」（描述的不是本表主鍵 → 搬家）

被問倒的地方直接抄進「優先修正」，修完再由原互評者確認一次。專題 Q&A 教師問的也是這五類。

## 專題進度建議（非繳交）

**到 U04，你的專題應該有「讀懂題目＋ER 圖＋schema 草稿」**。本單元建議把工作坊的成果收尾成「schema 定稿」：

1. **ER 圖一張**（拍照或 dbdiagram.io 匯出，之後直接貼進專題 notebook）——實體、關係、基數標清楚；
2. **可執行 DDL**：≥ 4 張表、PK/FK 齊、另外 ≥ 3 種約束；第一句 `PRAGMA foreign_keys = ON`；
3. **每表 ≥ 3 筆手工示範資料**（萬列合成資料是下個單元的事）；
4. **約束踩點測試 ≥ 4 個**（try/except，模仿 1.5 的第三格）；
5. **核心問題查詢 ≥ 5 句**：證明 schema 答得出你題目的關鍵問題（剩餘名額？是否衝堂？庫存夠不夠？）；
6. **設計決策筆記 3 條**（每條 2–3 行）：「為什麼 X 跟 Y 拆兩表」「這個 UNIQUE 對應需求哪句話」「快照/即算我選了哪個、為什麼」——報告的 Q&A 就從這裡出。

**不用繳交**——但下個單元（U05）課堂實作「把資料層接上 Gradio」會**直接用到你的 DDL**；帶著能跑的 schema 來，課堂 35 分鐘才夠用。
**與同學交流**：想法可以討論，但 DDL 各自寫、示範資料與查詢自己做——雷同且無法各自解釋，以抄襲處理。

# 本單元你應該帶走

1. 設計流程：情境 → 名詞動詞分析 → ER（基數靈魂拷問＋弱實體）→ 三規則轉 DDL。
2. 七大模式＝應用題目的樂高：主線跑 M:N＋屬性、區間重疊、狀態機、快照 vs 即算；事件流、時序價格、階層自參照為選讀，依題目查用。
3. 異常的病根：同一個事實存了 N 次；FD 是診斷工具，closure 機械化找 key（還能找出**全部**的 key）。
4. 3NF 實用心法：**一張表只講一個主題**；分解要無損（join 回來驗證）；（進階選讀）**BCNF 分解 vs 3NF 合成**——一個可能犧牲相依保持，一個保證保住。
5. 反正規化要有理由：報表快取（可重算）與歷史快照（不同的事實）；分析世界用星型＋dim_date。
6. AI 用兩輪：生成 → **反向質詢**；每條約束都要說得出對應需求的哪句話。

**下個單元**：把 schema 接上 Python——sqlite3 深入（`?` 傳值、交易、savepoint、date adapter）、萬列擬真資料合成，以及 **Gradio 初登場**＋★專題說明會（兩支示範影片）。讀物：Silberschatz ch6–7；Ullman ch3–4。

---
## 附錄 A：設計 checklist（schema 定稿前逐條打勾）

- [ ] 每張表一個主題，講得出「一列代表一個＿＿」
- [ ] 每張表有 PK（代理鍵 `INTEGER PRIMARY KEY` 是穩妥預設；自然鍵要真的永遠唯一）
- [ ] 每個 1:N 的 N 方有 FK；每個 M:N 有關聯表＋`UNIQUE(fk1, fk2)`
- [ ] 「同一 X 不可重複 Y」都有對應 UNIQUE
- [ ] 值域規則都有 CHECK（狀態、正數、日期先後）
- [ ] FK 的刪除策略想過了（預設擋下／CASCADE／SET NULL——弱實體用 CASCADE）
- [ ] 日期時間存 ISO 字串；金額用 INTEGER（元或分），避免 REAL 浮點誤差破壞加總與對帳
- [ ] 命名一致（英文 snake_case；主鍵 `表名_id`）
- [ ] 可推導的值不重複儲存——除非是「歷史快照」或講得出理由的快取
- [ ] `PRAGMA foreign_keys = ON` 寫在連線後第一句
- [ ] 用 3 個「核心問題查詢」實測過 schema 答得出來
- [ ] 跑過自檢器、跑過異常踩點測試

## 附錄 A2：schema 壞味道速查（互檢時的嗅覺）

| 壞味道 | 症狀 | 處方 |
|---|---|---|
| 上帝表（god table） | 一張表 20+ 欄、什麼都記 | 名詞動詞分析重拆（2.1 的下場） |
| 重複群組 | `tel1, tel2, tel3`／逗號清單／整包 JSON | 一列一值的子表（練習 F2） |
| 假日期 | 日期存 `10/5` 或民國年字串 | ISO `YYYY-MM-DD`（U02 的教訓） |
| 可推導欄氾濫 | 存了 `total = price*qty` 又存 price、qty | 即算；要快取就配對帳（模式⑤） |
| 狀態欄爆炸 | `is_paid, is_shipped, is_cancelled, ...` 一堆布林 | 一欄 status＋CHECK＋轉移表（模式③） |
| 幽靈外鍵 | 欄名叫 `xx_id` 但沒有 REFERENCES | 補 FK＋開 PRAGMA（U02） |
| 用名字當鍵 | 姓名、書名當 join 接點 | 代理鍵（2.5 反例的鬼列） |
| M:N 直連未落地 | 兩端互塞 FK，或一欄存一串 id | 關聯表＋兩支 FK（模式①） |
| EAV 萬用表 | `attr_name/attr_value` 承包所有核心欄位 | 穩定屬性回到型別化欄位；重複值拆子表 |



### A2.1 兩個最會偽裝成「彈性」的反模式：before → after

**① 把 ER 圖的 M:N 線直接抄成兩張表**。ER 概念圖畫 M:N 沒問題；錯在進入關聯 schema 時沒有落成關聯表。

```text
BEFORE ✗
student(student_id PK, course_ids TEXT)       # "C01,C07"：不能設逐筆 FK
course(course_id PK, student_id FK)           # 反向塞一支 FK 也只表達一個學生

AFTER ✓
student(student_id PK)
course(course_id PK)
enrollment(student_id FK, course_id FK, grade,
           PRIMARY KEY(student_id, course_id))
```

關聯本身若有成績、數量、狀態、時間，這些欄位只屬於 `enrollment`；放在任一端都會問出「到底是哪一門／哪一人？」。

**② EAV（Entity–Attribute–Value）把每個欄位都變成資料列**。

```text
BEFORE ✗
student_attr(student_id, attr_name, attr_value TEXT)
# (S001, 'entry_year', 'two thousand') 也能混進來；NOT NULL/CHECK/FK 各自失去落點

AFTER ✓
student(student_id PK, entry_year INTEGER CHECK(entry_year >= 2000), birth_date TEXT)
student_skill(student_id FK, skill_id FK, PRIMARY KEY(student_id, skill_id))
```

EAV 讓「加欄位」看似不用改 schema，代價是型別、必填、唯一性與查詢都轉嫁給每一段程式。**已知且會查／會約束的核心屬性不要用 EAV**。只有屬性真的由使用者動態定義時才考慮它，且至少要另有屬性定義表、型別與驗證策略；「懶得釐清需求」不是理由。

## 附錄 B：讀物地圖（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| §1 ER 模型、基數、弱實體 | ch6（§6.1–6.7） | §4.1–4.4 |
| ER → 關聯綱要 | §6.7 | §4.5–4.6 |
| §2 FD、closure、正規化 | ch7（§7.1–7.5） | §3.1–3.5 |
| BCNF vs 3NF、合成演算法 | §7.5–7.6 | §3.3、§3.5 |
| MVD／4NF | §7.7 | §3.6–3.7 |
| 星型 schema | §11.2（Data Warehousing） | §10.6 |